In [ ]:
# reorganize_and_explore.py
from pathlib import Path
import pandas as pd
import numpy as np
import keras
import matplotlib.pyplot as plt

# Use relative path from notebook location
# This works whether you're in notebooks/, scripts/, or project root
notebook_dir = Path.cwd()

# Find project root (where pyproject.toml is)
project_root = notebook_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:  # Reached filesystem root
        raise FileNotFoundError("Could not find project root (pyproject.toml not found)")

# Now use relative paths from project root
base_dir = project_root / "fall_detection_data"
processed_dir = base_dir / "processed"
models_dir = base_dir / "models"
models_dir.mkdir(exist_ok=True)
output_dir = models_dir

print(f"📂 Project root: {project_root}")
print(f"📂 Data directory: {base_dir}")
print(f"📂 Models directory: {models_dir}")
print()

print("=" * 80)
print("CURRENT DIRECTORY STRUCTURE")
print("=" * 80)

# Show current structure
for item in sorted(base_dir.iterdir()):
    if item.is_dir():
        print(f"\n📁 {item.name}/")
        # Show what's inside each directory
        sub_items = list(item.iterdir())[:5]
        for sub in sub_items:
            if sub.is_dir():
                file_count = len(list(sub.glob("*")))
                print(f"   📁 {sub.name}/ ({file_count} files)")
            else:
                print(f"   📄 {sub.name}")
        if len(list(item.iterdir())) > 5:
            print(f"   ... and {len(list(item.iterdir())) - 5} more")

print("\n" + "=" * 80)
print("PROPOSED REORGANIZATION")
print("=" * 80)

proposed_structure = """
fall_detection_data/
├── KFall/
│   ├── sensor_data/
│   │   ├── SA06/
│   │   │   ├── S06T01R01.csv  (KFall format: S##T##R##.csv)
│   │   │   ├── S06T02R01.csv
│   │   │   └── ...
│   │   └── SA07/ ...
│   └── labels/
│       ├── SA06_label.xlsx
│       └── SA07_label.xlsx ...
│
├── SisFall/
│   ├── SA01/
│   │   ├── D01_SA01_R01.txt  (SisFall format: <CODE>_<SUBJECT>_<TRIAL>.txt)
│   │   ├── F01_SA01_R01.txt
│   │   └── ...
│   ├── SA02/ ...
│   └── SE01/ ... (elderly subjects)
│
└── processed/
    ├── kfall_features.pkl
    ├── sisfall_features.pkl
    └── fused_dataset.pkl
"""

print(proposed_structure)

print("\n" + "=" * 80)
print("DATASET COMPARISON")
print("=" * 80)

# KFall structure
kfall_sensor = base_dir / "KFall" / "sensor_data"
if kfall_sensor.exists():
    kfall_subjects = sorted([d.name for d in kfall_sensor.iterdir() if d.is_dir()])
    sample_kfall = kfall_sensor / kfall_subjects[0]
    sample_kfall_file = list(sample_kfall.glob("*.csv"))[0]
    
    df_kfall = pd.read_csv(sample_kfall_file)
    
    print("\n📊 KFALL DATASET:")
    print(f"   Subjects: {len(kfall_subjects)} (SA06-SA38)")
    print(f"   Sampling Rate: 100 Hz (needs upsampling to 200 Hz)")
    print(f"   File Format: S##T##R##.csv")
    print(f"   Columns: {df_kfall.columns.tolist()}")
    print(f"   Data Shape (sample): {df_kfall.shape}")
    print(f"   Has Labels: ✅ Yes (temporal annotations in Excel files)")

# SisFall structure
sisfall_dir = base_dir / "SisFall"
if sisfall_dir.exists():
    sisfall_subjects = sorted([d.name for d in sisfall_dir.iterdir() if d.is_dir()])
    adults = [s for s in sisfall_subjects if s.startswith('SA')]
    elderly = [s for s in sisfall_subjects if s.startswith('SE')]
    
    sample_sisfall = sisfall_dir / adults[0]
    sample_sisfall_file = list(sample_sisfall.glob("*.txt"))[0]
    
    # Read SisFall file - more robust parsing
    try:
        # Method 1: Read line by line and parse manually
        with open(sample_sisfall_file, 'r') as f:
            lines = f.readlines()
        
        data = []
        for line in lines:
            # Remove semicolon and split by comma or whitespace
            line = line.strip().replace(';', '')
            values = line.replace(',', ' ').split()
            if len(values) == 9:  # Should have 9 columns
                data.append([float(v) for v in values])
        
        df_sisfall = pd.DataFrame(data)
        
        print("\n📊 SISFALL DATASET:")
        print(f"   Subjects: {len(sisfall_subjects)} total")
        print(f"     - Adults (SA): {len(adults)} (SA01-SA23)")
        print(f"     - Elderly (SE): {len(elderly)} (SE01-SE15)")
        print(f"   Sampling Rate: 200 Hz ✅")
        print(f"   File Format: <CODE>_<SUBJECT>_<TRIAL>.txt")
        print(f"   Columns: 9 (ADXL345: 0-2, ITG3200: 3-5, MMA8451Q: 6-8)")
        print(f"   Data Shape (sample): {df_sisfall.shape}")
        print(f"   Has Labels: ❌ No (must use Algorithm 1)")
        print(f"   Data Format: Raw bits (needs conversion to physical units)")
        
    except Exception as e:
        print(f"\n❌ Error reading SisFall file: {e}")
        print("   Will handle this in the preprocessing pipeline")

print("\n" + "=" * 80)
print("ACTIVITIES NEEDED FOR PAPER REPRODUCTION")
print("=" * 80)

print("\n📋 FROM KFALL (Table I):")
kfall_needed = {
    'T10': 'Stumble while walking',
    'T28': 'Vertical fall while walking (fainting)',
    'T30': 'Forward fall while walking (trip)',
    'T31': 'Forward fall while jogging (trip)',
    'T32': 'Forward fall while walking (slip)',
    'T33': 'Lateral fall while walking (slip)',
    'T34': 'Backward fall while walking (slip)'
}
for code, desc in kfall_needed.items():
    print(f"   {code}: {desc}")

print("\n📋 FROM SISFALL (Table I):")
print("\n   ADL Activities:")
sisfall_adl = {
    'D01': 'Walking slowly',
    'D02': 'Walking quickly',
    'D03': 'Jogging slowly',
    'D04': 'Jogging quickly',
    'D05': 'Walking upstairs/downstairs slowly',
    'D06': 'Walking upstairs/downstairs quickly',
    'D18': 'Stumble while walking'
}
for code, desc in sisfall_adl.items():
    print(f"   {code}: {desc}")

print("\n   Fall Activities:")
sisfall_falls = {
    'F01': 'Fall forward while walking (slip)',
    'F02': 'Fall backward while walking (slip)',
    'F03': 'Lateral fall while walking (slip)',
    'F04': 'Fall forward while walking (trip)',
    'F05': 'Fall forward while jogging (trip)',
    'F06': 'Vertical fall while walking (fainting)'
}
for code, desc in sisfall_falls.items():
    print(f"   {code}: {desc}")

print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("""
1. ✅ Data is properly organized
2. ⏭️  Implement preprocessing pipeline:
   - Load and convert SisFall raw bits to physical units
   - Upsample KFall from 100Hz to 200Hz
   - Apply Algorithm 1 for temporal segmentation
   - Extract features according to Table I
3. ⏭️  Z-score normalization and dataset fusion
4. ⏭️  Build and train FallNet""")


In [ ]:
import sys
print(sys.executable)

In [ ]:
# Load the newly processed data
X_data = np.load(processed_dir / "X_data.npy")
y_labels = np.load(processed_dir / "y_labels.npy")

# ============================================================================
# STEP 1: Merge Impact and Aftermath
# ============================================================================
print("Merging Impact and Aftermath classes...")
y_labels[y_labels == 7] = 6  # Change Aftermath (7) to Impact (6)

# ============================================================================
# STEP 2: Remove Fall_Recovery (NEW!)
# ============================================================================
print("\n" + "="*80)
print("REMOVING FALL_RECOVERY CLASS")
print("="*80)

from collections import Counter

# Show before
counts_before = Counter(y_labels)
print(f"\nBefore removal:")
print(f"  Total samples: {len(y_labels):,}")
print(f"  Fall_Recovery (class 4): {counts_before[4]} samples")

# Remove Fall_Recovery (class 4)
mask = y_labels != 4
X_data = X_data[mask]
y_labels_temp = y_labels[mask]

removed_count = (~mask).sum()
print(f"\n✅ Removed {removed_count} Fall_Recovery samples")

# Shift labels down (5→4, 6→5)
y_labels = y_labels_temp.copy()
y_labels[y_labels_temp > 4] -= 1  # Classes 5,6 become 4,5

print(f"\nAfter removal:")
print(f"  Total samples: {len(y_labels):,}")
print(f"  Removed: {removed_count} samples ({removed_count/(len(y_labels)+removed_count)*100:.2f}%)")

# ============================================================================
# STEP 3: Update label map (NOW 6 CLASSES: 0-5)
# ============================================================================
label_map = {
    'Walking': 0,
    'Jogging': 1,
    'Walking_stairs_updown': 2,
    'Stumble_while_walking': 3,
    'Fall_Initiation': 4,      # Was 5, now 4 ← SHIFTED DOWN!
    'Impact_Aftermath': 5,     # Was 6, now 5 ← SHIFTED DOWN!
}
reverse_label_map = {v: k for k, v in label_map.items()}

print(f"\n✅ Updated to 6 classes (0-5):")
for name, idx in sorted(label_map.items(), key=lambda x: x[1]):
    print(f"  Class {idx}: {name}")

y_categorical = keras.utils.to_categorical(y_labels, num_classes=6)  # ← HERE!
print(f"y_categorical shape: {y_categorical.shape}")

# ============================================================================
# DIAGNOSTICS
# ============================================================================
print("\n" + "="*80)
print("POST-REMOVAL DATA DIAGNOSTICS")
print("="*80)

# 1. Class distribution
class_counts = Counter(y_labels)
print("\n1. Class Distribution (6 classes):")
for cls_idx in sorted(class_counts.keys()):
    count = class_counts[cls_idx]
    pct = count / len(y_labels) * 100
    print(f"   Class {cls_idx} ({reverse_label_map[cls_idx]:30s}): {count:5d} ({pct:5.2f}%)")

# Calculate imbalance
max_count = max(class_counts.values())
min_count = min(class_counts.values())
print(f"\nImbalance ratio: {max_count/min_count:.2f}x (was 36.8x with Fall_Recovery)")

# 2. Per-class signal statistics
print("\n2. Per-Class Signal Statistics (Acc-Y axis):")
print(f"   {'Class':<35s} {'Mean':<10s} {'Std':<10s} {'Min':<10s} {'Max':<10s}")
print(f"   {'-'*75}")
for cls_idx in sorted(class_counts.keys()):
    class_samples = X_data[y_labels == cls_idx]
    acc_y = class_samples[:, :, 1]  # Y-axis acceleration
    
    mean_val = acc_y.mean()
    std_val = acc_y.std()
    min_val = acc_y.min()
    max_val = acc_y.max()
    
    print(f"   {reverse_label_map[cls_idx]:<35s} {mean_val:>8.4f}  {std_val:>8.4f}  {min_val:>8.2f}  {max_val:>8.2f}")

# 3. Variance ranking
print("\n3. Variance Ranking (Fall_Initiation should be #1):")
variances = []
for cls_idx in sorted(class_counts.keys()):
    class_samples = X_data[y_labels == cls_idx]
    acc_y_var = class_samples[:, :, 1].var()
    variances.append((reverse_label_map[cls_idx], acc_y_var, cls_idx))
variances.sort(key=lambda x: x[1], reverse=True)
for i, (name, var, idx) in enumerate(variances, 1):
    print(f"   {i}. {name:<35s}: {var:.4f}")

# 4. Visualize samples (update to 6 classes)
fig, axes = plt.subplots(3, 2, figsize=(15, 10))
axes = axes.flatten()
critical_classes = [
    label_map['Walking'],
    label_map['Fall_Initiation'],
    label_map['Impact_Aftermath'],
    label_map['Stumble_while_walking'],
    label_map['Jogging'],
    label_map['Walking_stairs_updown']
]
for i, cls_idx in enumerate(critical_classes):
    if cls_idx in class_counts:
        sample_idx = np.where(y_labels == cls_idx)[0][0]
        sample_data = X_data[sample_idx]
        
        time = np.arange(200) / 200
        axes[i].plot(time, sample_data[:, 0], label='Acc-X', alpha=0.7, linewidth=1)
        axes[i].plot(time, sample_data[:, 1], label='Acc-Y', alpha=0.7, linewidth=1)
        axes[i].plot(time, sample_data[:, 2], label='Acc-Z', alpha=0.7, linewidth=1)
        
        axes[i].set_title(f'{reverse_label_map[cls_idx]}', fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Time (s)')
        axes[i].set_ylabel('Normalized Acc')
        axes[i].legend(fontsize=8)
        axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("✅ DATA READY FOR TRAINING (6 CLASSES)")
print("="*80)

In [ ]:
# %% [markdown]
# # Save Preprocessed 6-Class Data
# Save the cleaned dataset after merging Impact/Aftermath and removing Fall_Recovery

# %%
import numpy as np
from pathlib import Path

# Setup paths (same as before)
base_dir = Path("~/repos/summerschool2023/projects/fall-detection/fall_detection_data").expanduser()
processed_dir = base_dir / "processed"

print("="*80)
print("SAVING PREPROCESSED 6-CLASS DATA")
print("="*80)

# Save the processed data
save_path_X = processed_dir / "X_data_6class.npy"
save_path_y = processed_dir / "y_labels_6class.npy"
save_path_y_cat = processed_dir / "y_categorical_6class.npy"

np.save(save_path_X, X_data)
np.save(save_path_y, y_labels)
np.save(save_path_y_cat, y_categorical)

print(f"\n✅ Saved preprocessed data:")
print(f"   X_data:        {save_path_X}")
print(f"   y_labels:      {save_path_y}")
print(f"   y_categorical: {save_path_y_cat}")

print(f"\nSaved shapes:")
print(f"   X_data:        {X_data.shape}")
print(f"   y_labels:      {y_labels.shape}")
print(f"   y_categorical: {y_categorical.shape}")

# Also save the label mapping for future reference
label_map_path = processed_dir / "label_map_6class.npy"
np.save(label_map_path, label_map)
     # ✅ Labels (numbers 0-5)

# Save the LABEL MAPPING (dictionary)
import json
with open(processed_dir / "label_map_6class.json", 'w') as f:
    json.dump(label_map, f, indent=2)                          # ✅ Class names → numbers
print(f"\n   label_map:     {label_map_path}")

print("\n" + "="*80)
print("✅ ALL DATA SAVED SUCCESSFULLY")
print("="*80)
print("\nTo load this data in future notebooks:")
print("```python")
print("X_data = np.load(processed_dir / 'X_data_6class.npy')")
print("y_labels = np.load(processed_dir / 'y_labels_6class.npy')")
print("y_categorical = np.load(processed_dir / 'y_categorical_6class.npy')")
print("label_map = np.load(processed_dir / 'label_map_6class.npy', allow_pickle=True).item()")
print("```")

In [ ]:
# %% [markdown]
# # FallNet Training Pipeline
# CNN-LMU ensemble for fall detection with 6 classes

# %%
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')
from keras_lmu import LMU

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# %% [markdown]
## 1. FallNet Model Architecture

# %%
class FallNet:
    """
    FallNet: CNN-LmU Ensemble for Pre-Impact Fall Detection
    """
    
    def __init__(self, input_shape=(200, 6), n_classes=6):
        """
        Args:
            input_shape: (timesteps, features) = (200, 6)
            n_classes: Number of output classes (6)
        """
        self.input_shape = input_shape
        self.n_classes = n_classes
        self.model = None
    
    def build_lmu_branch(self, inputs):
        """LMU Branch"""
        x = LMU(
            memory_d=32,      # REDUCED from 64
            order=32,          # REDUCED from 64
            theta=200.0,
            hidden_cell=None,
            kernel_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),  # INCREASED from 1e-4
            recurrent_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),
            dropout=0.3,       # NEW
            recurrent_dropout=0.2,  # NEW
            name='lmu'
            )(inputs)
    
        
        # Dense layers with stronger regularization
        x = layers.Dense(
            128,
            activation='relu',
            kernel_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),
            name='lmu_dense1'
            )(x)
        x = layers.Dropout(0.4, name='lmu_dropout1')(x)  # INCREASED
    
        x = layers.Dense(
            64,
            activation='relu',
            kernel_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),
            name='lmu_dense2'
            )(x)
        x = layers.Dropout(0.3, name='lmu_dropout2')(x)  # INCREASED
    
        x = layers.Dense(
            32,
            activation='relu',
            kernel_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),
            name='lmu_dense3'
            )(x)
        x = layers.Dropout(0.2, name='lmu_dropout3')(x)
    
        lmu_output = layers.Dense(
            self.n_classes,
            activation='softmax',
            name='lmu_output'
            )(x)
    
        return lmu_output
        

    def build_lstm_branch(self, inputs):
        """LSTM Branch"""
        x = layers.LSTM(
            units=256,
            activation='tanh',
            return_sequences=False,
            name='lstm_layer'
        )(inputs)
        
        x = layers.Dense(128, activation='relu', name='lstm_dense1')(x)
        x = layers.Dropout(0.2, name='lstm_dropout1')(x)
        
        x = layers.Dense(64, activation='relu', name='lstm_dense2')(x)
        x = layers.Dropout(0.2, name='lstm_dropout2')(x)
        
        x = layers.Dense(32, activation='relu', name='lstm_dense3')(x)
        x = layers.Dropout(0.2, name='lstm_dropout3')(x)
        
        lstm_output = layers.Dense(
            self.n_classes, 
            activation='softmax',
            name='lstm_output'
        )(x)
        
        return lstm_output
    
    def build_cnn_branch(self, inputs):
        """CNN Branch"""
        x = layers.Conv1D(
            filters=128,
            kernel_size=3,
            activation='relu',
            padding='same',
            name='conv1d_layer'
        )(inputs)
        
        x = layers.MaxPooling1D(pool_size=2, name='maxpool_layer')(x)
        x = layers.Flatten(name='flatten_layer')(x)
        
        x = layers.Dense(1024, activation='relu', name='cnn_dense1')(x)
        x = layers.Dropout(0.2, name='cnn_dropout1')(x)
        
        x = layers.Dense(512, activation='relu', name='cnn_dense2')(x)
        x = layers.Dropout(0.2, name='cnn_dropout2')(x)
        
        cnn_output = layers.Dense(
            self.n_classes,
            activation='softmax',
            name='cnn_output'
        )(x)
        
        return cnn_output

    def build_cnn_only(self):
        """Build CNN-only model (no temporal component)"""
        inputs = layers.Input(shape=self.input_shape, name='input')
        cnn_output = self.build_cnn_branch(inputs)
        
        self.model = models.Model(
            inputs=inputs,
            outputs=cnn_output,
            name='FallNet_CNN_Only'
        )
        return self.model
    
    def build_lstm_only(self):
        """Build LSTM-only model (temporal encoding via gates)"""
        inputs = layers.Input(shape=self.input_shape, name='input')
        lstm_output = self.build_lstm_branch(inputs)
        
        self.model = models.Model(
            inputs=inputs,
            outputs=lstm_output,
            name='FallNet_LSTM_Only'
        )
        return self.model
    
    def build_lmu_only(self):
        """Build LMU-only model (temporal encoding via Legendre polynomials)"""
        inputs = layers.Input(shape=self.input_shape, name='input')
        lmu_output = self.build_lmu_branch(inputs)
        
        self.model = models.Model(
            inputs=inputs,
            outputs=lmu_output,
            name='FallNet_LMU_Only'
        )
        return self.model
    
    def build_ensemble(self):
        """Build the complete ensemble model"""
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        lmu_output = self.build_lstm_branch(inputs)
        cnn_output = self.build_cnn_branch(inputs)
        
        ensemble_output = layers.Average(name='ensemble_average')([lmu_output, cnn_output])
        
        self.model = models.Model(
            inputs=inputs,
            outputs=ensemble_output,
            name='FallNet_CNN_LSTM'
        )
        
        return self.model
    
    def compile_model(self, learning_rate=None):
        """Compile model"""
        if self.model is None:
            raise ValueError("Model not built yet. Call build_ensemble() first.")
        
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate) if learning_rate else keras.optimizers.Adam()
        
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # 100x smaller
            loss='categorical_crossentropy',
            metrics=[
                'accuracy', 
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall')
            ]
        )
        
        return self.model

print("✅ FallNet class defined")

# %% [markdown]
## 2. Build and Display Model

# %%
print("\n" + "="*80)
print("BUILDING FALLNET MODEL")
print("="*80)

# Create instance with 6 classes
fallnet = FallNet(input_shape=(200, 6), n_classes=6)

# Build ensemble
model = fallnet.build_ensemble()

# Compile
model = fallnet.compile_model()

# Display architecture
print("\n")
model.summary()

# Count parameters
def count_parameters(model):
    trainable = np.sum([np.prod(v.shape) for v in model.trainable_weights])
    non_trainable = np.sum([np.prod(v.shape) for v in model.non_trainable_weights])
    return trainable, non_trainable

trainable, non_trainable = count_parameters(model)

print("\n" + "="*80)
print("MODEL PARAMETERS")
print("="*80)
print(f"Trainable:     {trainable:,}")
print(f"Non-trainable: {non_trainable:,}")
print(f"Total:         {trainable + non_trainable:,}")

# %% [markdown]
## 3. Training Configuration

# %%
BATCH_SIZE = 2
EPOCHS = 50
K_FOLDS = 5

print("\n" + "="*80)
print("TRAINING CONFIGURATION")
print("="*80)
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {EPOCHS}")
print(f"K-Folds:    {K_FOLDS}")
print(f"Using data from previous cell (6 classes, {len(y_labels):,} samples)")

# %% [markdown]
## 4. Verify Data Before Training

# %%
print("\n" + "="*80)
print("PRE-TRAINING VERIFICATION")
print("="*80)

print(f"✅ Data shapes:")
print(f"   X_data:        {X_data.shape}")
print(f"   y_labels:      {y_labels.shape}")
print(f"   y_categorical: {y_categorical.shape}")
print(f"\n✅ Classes: {len(np.unique(y_labels))} (should be 6)")
print(f"✅ Label range: {y_labels.min()}-{y_labels.max()} (should be 0-5)")
print(f"✅ Model output: {model.output_shape[-1]} (should be 6)")

assert X_data.shape[0] == y_labels.shape[0] == y_categorical.shape[0], "Shape mismatch!"
assert len(np.unique(y_labels)) == 6, "Should have 6 classes!"
assert y_labels.max() == 5, "Max label should be 5!"
assert model.output_shape[-1] == 6, "Model should output 6 classes!"

print("\n✅ All checks passed - ready to train!")

# %% [markdown]
## 5. K-Fold Cross-Validation Training

# %%
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_results = []
fold_histories = []

print("\n" + "="*80)
print("STARTING K-FOLD CROSS-VALIDATION")
print("="*80)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"{'='*80}")
    
    # Split data
    X_train, X_val = X_data[train_idx], X_data[val_idx]
    y_train, y_val = y_categorical[train_idx], y_categorical[val_idx]
    
    print(f"Train: {X_train.shape[0]:,} samples | Val: {X_val.shape[0]:,} samples")
    
    # Build fresh model for this fold
    fallnet_fold = FallNet(input_shape=(200, 6), n_classes=6)
    model_fold = fallnet_fold.build_ensemble()
    model_fold = fallnet_fold.compile_model()
    
    # Define callbacks for THIS fold
    fold_callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=20,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=10,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=str(output_dir / f'fallnet_lstm_fold_{fold}.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
    ]
    # %% [markdown]
## 5.5 Calculate Class Weights

# %%
    from sklearn.utils.class_weight import compute_class_weight

    print("\n" + "="*80)
    print("CALCULATING CLASS WEIGHTS")
    print("="*80)

# Calculate balanced weights
    class_weights_array = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_labels),
        y=y_labels
    )

# Cap at 3x to prevent training instability
    MAX_WEIGHT = 3.0
    class_weights_array_capped = np.clip(class_weights_array, None, MAX_WEIGHT)
    class_weights = dict(enumerate(class_weights_array_capped))

    print("\nClass Distribution:")
    from collections import Counter
    counts = Counter(y_labels)
    for cls_idx in range(6):
        count = counts[cls_idx]
        pct = count / len(y_labels) * 100
        weight = class_weights[cls_idx]
        print(f"  {reverse_label_map[cls_idx]:<30s}: {count:>5d} ({pct:>5.2f}%) → weight: {weight:.2f}x")

    print(f"\n✅ Weight range: {min(class_weights.values()):.2f}x to {max(class_weights.values()):.2f}x")
    print(f"✅ Max/Min ratio: {max(class_weights.values())/min(class_weights.values()):.2f}x (was 4.0x without capping)")
    # Train WITHOUT class weights
    print(f"\nTraining fold {fold}...")
    history = model_fold.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        class_weight=class_weights,  # ← ADD THIS LINE!
        callbacks=fold_callbacks,
        verbose=1
    )
    
    # Evaluate
    val_loss, val_acc, val_precision, val_recall = model_fold.evaluate(X_val, y_val, batch_size=2, verbose=0)
    val_f1 = 2 * (val_precision * val_recall) / (val_precision + val_recall) if (val_precision + val_recall) > 0 else 0
    
    print(f"\n{'='*50}")
    print(f"Fold {fold} Results:")
    print(f"{'='*50}")
    print(f"Loss:      {val_loss:.4f}")
    print(f"Accuracy:  {val_acc:.4f}")
    print(f"Precision: {val_precision:.4f}")
    print(f"Recall:    {val_recall:.4f}")
    print(f"F1-Score:  {val_f1:.4f}")
    
    # Store results
    fold_results.append({
        'fold': fold,
        'val_loss': val_loss,
        'val_accuracy': val_acc,
        'val_precision': val_precision,
        'val_recall': val_recall,
        'val_f1': val_f1
    })
    
    fold_histories.append(history.history)
    
    print(f"✅ Model saved: fallnet_lstm_fold_{fold}.keras")

print("\n" + "="*80)
print("K-FOLD CROSS-VALIDATION COMPLETE")
print("="*80)

# %% [markdown]
## 6. Aggregate Results

# %%
results_df = pd.DataFrame(fold_results)

print("\n" + "="*80)
print("RESULTS ACROSS ALL FOLDS")
print("="*80)
print(results_df.to_string(index=False))

print("\n" + "="*80)
print("AVERAGE PERFORMANCE ± STD")
print("="*80)

mean_results = results_df.mean(numeric_only=True)
std_results = results_df.std(numeric_only=True)

metrics_table = []
for metric in ['val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1']:
    metrics_table.append({
        'Metric': metric,
        'Mean': f"{mean_results[metric]:.4f}",
        'Std': f"±{std_results[metric]:.4f}"
    })

metrics_df = pd.DataFrame(metrics_table)
print(metrics_df.to_string(index=False))

# %% [markdown]
## 7. Visualize Training History

# %%
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = [
    ('loss', 'Loss'),
    ('accuracy', 'Accuracy'),
    ('precision', 'Precision'),
    ('recall', 'Recall')
]

for idx, (metric, title) in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    for fold, history in enumerate(fold_histories, 1):
        epochs = range(1, len(history[metric]) + 1)
        ax.plot(epochs, history[metric], label=f'Fold {fold} Train', alpha=0.5, linewidth=1)
        ax.plot(epochs, history[f'val_{metric}'], label=f'Fold {fold} Val', 
                linestyle='--', alpha=0.7, linewidth=1.5)
    
    ax.set_title(f'{title} Across All Folds', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('FallNet Training History - 5-Fold Cross-Validation', 
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(output_dir / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Training history saved to {output_dir / 'training_history.png'}")

# %% [markdown]
## 8. Detailed Evaluation on Best Fold

# %%
best_fold = int(results_df.loc[results_df['val_f1'].idxmax(), 'fold'])

print("\n" + "="*80)
print(f"DETAILED EVALUATION - BEST FOLD #{best_fold}")
print("="*80)
print(f"Best fold F1-Score: {results_df.loc[results_df['fold']==best_fold, 'val_f1'].values[0]:.4f}")

# Load best model
best_model = keras.models.load_model(output_dir / f'fallnet_fold_lstm_{best_fold}.keras')

# Get predictions on ALL data
y_pred_probs = best_model.predict(X_data, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report
class_names = [reverse_label_map[i] for i in range(6)]

print("\n" + "="*80)
print("CLASSIFICATION REPORT (Best Fold on All Data)")
print("="*80)
print(classification_report(y_labels, y_pred, target_names=class_names, digits=4))

# %% [markdown]
## 9. Per-Class Detailed Metrics

# %%
print("\n" + "="*80)
print("PER-CLASS DETAILED METRICS")
print("="*80)

print(f"\n{'Class':<40s} {'Precision':<12s} {'Recall':<12s} {'F1-Score':<12s} {'Support'}")
print("-"*90)

for cls_idx in range(6):
    precision = precision_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    recall = recall_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    f1 = f1_score(y_labels == cls_idx, y_pred == cls_idx, zero_division=0)
    support = np.sum(y_labels == cls_idx)
    
    print(f"{reverse_label_map[cls_idx]:<40s} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {support}")

# %% [markdown]
## 10. Confusion Matrix

# %%
cm = confusion_matrix(y_labels, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix - Best Fold (6 Classes)', fontsize=15, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(output_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Confusion matrix saved to {output_dir / 'confusion_matrix.png'}")

# %% [markdown]
## 11. Final Summary

# %%
# Get Fall_Initiation metrics
fall_init_idx = label_map["Fall_Initiation"]
fall_init_precision = precision_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
fall_init_recall = recall_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
fall_init_f1 = f1_score(y_labels == fall_init_idx, y_pred == fall_init_idx)

print("\n" + "="*80)
print("TRAINING COMPLETE - FINAL SUMMARY")
print("="*80)

summary = f"""
✅ Successfully trained FallNet with 5-fold cross-validation

Configuration:
  - Model: CNN-LSTM Ensemble (6 classes)
  - Total samples: {len(y_labels):,}
  - Training samples per fold: ~{len(y_labels)*0.8//K_FOLDS:,.0f}
  - Validation samples per fold: ~{len(y_labels)*0.2//K_FOLDS:,.0f}

Average Performance (5-fold CV):
  - Accuracy:  {mean_results['val_accuracy']:.4f} ± {std_results['val_accuracy']:.4f}
  - Precision: {mean_results['val_precision']:.4f} ± {std_results['val_precision']:.4f}
  - Recall:    {mean_results['val_recall']:.4f} ± {std_results['val_recall']:.4f}
  - F1-Score:  {mean_results['val_f1']:.4f} ± {std_results['val_f1']:.4f}

Fall_Initiation Performance (Critical Class):
  - Recall (Sensitivity): {fall_init_recall:.4f}
  - F1-Score:             {fall_init_f1:.4f}

Saved Files:
  - Training history:    {output_dir / 'training_history.png'}
  - Confusion matrix:    {output_dir / 'confusion_matrix.png'}
  - Best model:          {output_dir / f'fallnet_fold_{best_fold}.keras'}
  - All fold models:     {output_dir / 'fallnet_fold_*.keras'}
"""

print(summary)

with open(output_dir / 'training_summary.txt', 'w') as f:
    f.write(summary)

print(f"✅ Summary saved to {output_dir / 'training_summary.txt'}")

In [ ]:
# %% [markdown]
# # Data Loading and Preprocessing
# Load preprocessed data, merge classes, remove Fall_Recovery

# %%
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path
from tensorflow import keras
import json

# ========================================================================
# Setup paths (portable)
# ========================================================================
current_dir = Path.cwd()

# Find project root by looking for pyproject.toml
project_root = current_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:  # Reached filesystem root
        # Fallback: assume we're in notebooks/ directory
        project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
        break

# Define paths relative to project root
base_dir = project_root / "fall_detection_data"
processed_dir = base_dir / "processed"
output_dir = base_dir / "models"
output_dir.mkdir(exist_ok=True)

print("="*80)
print("LOADING AND PREPROCESSING DATA")
print("="*80)
print(f"\n📂 Project root: {project_root}")
print(f"📂 Data directory: {base_dir}")
print(f"📂 Processed directory: {processed_dir}")
print(f"📂 Models directory: {output_dir}")

# %% Load the newly processed data
X_data = np.load(processed_dir / "X_data.npy")
y_labels = np.load(processed_dir / "y_labels.npy")

print(f"\nOriginal data loaded:")
print(f"  X_data shape: {X_data.shape}")
print(f"  y_labels shape: {y_labels.shape}")

# ============================================================================
# STEP 1: Merge Impact and Aftermath
# ============================================================================
print("\n" + "="*80)
print("STEP 1: MERGING IMPACT AND AFTERMATH")
print("="*80)

counts_before_merge = Counter(y_labels)
print(f"Before merge:")
print(f"  Impact (6): {counts_before_merge[6]} samples")
print(f"  Aftermath (7): {counts_before_merge[7]} samples")

y_labels[y_labels == 7] = 6  # Change Aftermath (7) to Impact (6)

counts_after_merge = Counter(y_labels)
print(f"\nAfter merge:")
print(f"  Impact_Aftermath (6): {counts_after_merge[6]} samples")

# ============================================================================
# STEP 2: Remove Fall_Recovery
# ============================================================================
print("\n" + "="*80)
print("STEP 2: REMOVING FALL_RECOVERY CLASS")
print("="*80)

# Show before
counts_before = Counter(y_labels)
print(f"\nBefore removal:")
print(f"  Total samples: {len(y_labels):,}")
print(f"  Fall_Recovery (class 4): {counts_before[4]} samples")

# Remove Fall_Recovery (class 4)
mask = y_labels != 4
X_data = X_data[mask]
y_labels_temp = y_labels[mask]

removed_count = (~mask).sum()
print(f"\n✅ Removed {removed_count} Fall_Recovery samples")

# Shift labels down (5→4, 6→5)
y_labels = y_labels_temp.copy()
y_labels[y_labels_temp > 4] -= 1  # Classes 5,6 become 4,5

print(f"\nAfter removal:")
print(f"  Total samples: {len(y_labels):,}")
print(f"  Removed: {removed_count} samples ({removed_count/(len(y_labels)+removed_count)*100:.2f}%)")

# ============================================================================
# STEP 3: Update label map (NOW 6 CLASSES: 0-5)
# ============================================================================
print("\n" + "="*80)
print("STEP 3: CREATING 6-CLASS LABEL MAP")
print("="*80)

label_map = {
    'Walking': 0,
    'Jogging': 1,
    'Walking_stairs_updown': 2,
    'Stumble_while_walking': 3,
    'Fall_Initiation': 4,      # Was 5, now 4
    'Impact_Aftermath': 5,     # Was 6, now 5
}
reverse_label_map = {v: k for k, v in label_map.items()}

print(f"\n✅ Updated to 6 classes (0-5):")
for name, idx in sorted(label_map.items(), key=lambda x: x[1]):
    print(f"  Class {idx}: {name}")

# ============================================================================
# STEP 4: Create categorical labels
# ============================================================================
print("\n" + "="*80)
print("STEP 4: CREATING CATEGORICAL LABELS")
print("="*80)

y_categorical = keras.utils.to_categorical(y_labels, num_classes=6)

print(f"\n✅ y_categorical created:")
print(f"   Shape: {y_categorical.shape}")
print(f"   Expected: ({len(y_labels)}, 6)")

# Verification
assert X_data.shape[0] == y_labels.shape[0] == y_categorical.shape[0], \
    "Data shapes don't match!"

print(f"\n✅ All data aligned:")
print(f"   X_data:        {X_data.shape}")
print(f"   y_labels:      {y_labels.shape}")
print(f"   y_categorical: {y_categorical.shape}")
print(f"   All have {X_data.shape[0]:,} samples")

# ============================================================================
# STEP 5: SAVE 6-CLASS DATA
# ============================================================================
print("\n" + "="*80)
print("STEP 5: SAVING 6-CLASS DATA")
print("="*80)

# Save arrays with 6class suffix
np.save(processed_dir / "X_data_6class.npy", X_data)
np.save(processed_dir / "y_labels_6class.npy", y_labels)

# Save label map
with open(processed_dir / "label_map_6class.json", 'w') as f:
    json.dump(label_map, f, indent=2)

print(f"\n✅ Saved 6-class data:")
print(f"   {processed_dir / 'X_data_6class.npy'}")
print(f"   {processed_dir / 'y_labels_6class.npy'}")
print(f"   {processed_dir / 'label_map_6class.json'}")

# ============================================================================
# DIAGNOSTICS
# ============================================================================
print("\n" + "="*80)
print("DATA QUALITY DIAGNOSTICS")
print("="*80)

# 1. Class distribution
class_counts = Counter(y_labels)
print("\n1. Class Distribution (6 classes):")
for cls_idx in sorted(class_counts.keys()):
    count = class_counts[cls_idx]
    pct = count / len(y_labels) * 100
    print(f"   Class {cls_idx} ({reverse_label_map[cls_idx]:30s}): {count:5d} ({pct:5.2f}%)")

# Calculate imbalance
max_count = max(class_counts.values())
min_count = min(class_counts.values())
print(f"\nImbalance ratio: {max_count/min_count:.2f}x")
print(f"  (was 36.8x with Fall_Recovery)")

# 2. Per-class signal statistics
print("\n2. Per-Class Signal Statistics (Acc-Y axis):")
print(f"   {'Class':<35s} {'Mean':<10s} {'Std':<10s} {'Min':<10s} {'Max':<10s}")
print(f"   {'-'*75}")
for cls_idx in sorted(class_counts.keys()):
    class_samples = X_data[y_labels == cls_idx]
    acc_y = class_samples[:, :, 1]  # Y-axis acceleration
    
    mean_val = acc_y.mean()
    std_val = acc_y.std()
    min_val = acc_y.min()
    max_val = acc_y.max()
    
    print(f"   {reverse_label_map[cls_idx]:<35s} {mean_val:>8.4f}  {std_val:>8.4f}  {min_val:>8.2f}  {max_val:>8.2f}")

# 3. Variance ranking
print("\n3. Variance Ranking (Fall_Initiation should be #1 or #2):")
variances = []
for cls_idx in sorted(class_counts.keys()):
    class_samples = X_data[y_labels == cls_idx]
    acc_y_var = class_samples[:, :, 1].var()
    variances.append((reverse_label_map[cls_idx], acc_y_var, cls_idx))
variances.sort(key=lambda x: x[1], reverse=True)

for i, (name, var, idx) in enumerate(variances, 1):
    marker = "✅" if name == "Fall_Initiation" and i <= 2 else "  "
    print(f"   {marker} {i}. {name:<35s}: {var:.4f}")

# Check Fall_Initiation rank
fall_init_rank = next(i for i, (name, _, _) in enumerate(variances, 1) if name == "Fall_Initiation")
if fall_init_rank <= 2:
    print(f"\n✅ PASS: Fall_Initiation ranked #{fall_init_rank} (should be ≤2)")
else:
    print(f"\n⚠️  WARNING: Fall_Initiation ranked #{fall_init_rank} (should be ≤2)")

# 4. Visualize samples
print("\n4. Sample Visualization:")
fig, axes = plt.subplots(3, 2, figsize=(15, 10))
axes = axes.flatten()
critical_classes = [
    label_map['Walking'],
    label_map['Fall_Initiation'],
    label_map['Impact_Aftermath'],
    label_map['Stumble_while_walking'],
    label_map['Jogging'],
    label_map['Walking_stairs_updown']
]

for i, cls_idx in enumerate(critical_classes):
    if cls_idx in class_counts:
        sample_idx = np.where(y_labels == cls_idx)[0][0]
        sample_data = X_data[sample_idx]
        
        time = np.arange(200) / 200
        axes[i].plot(time, sample_data[:, 0], label='Acc-X', alpha=0.7, linewidth=1)
        axes[i].plot(time, sample_data[:, 1], label='Acc-Y', alpha=0.7, linewidth=1)
        axes[i].plot(time, sample_data[:, 2], label='Acc-Z', alpha=0.7, linewidth=1)
        
        axes[i].set_title(f'{reverse_label_map[cls_idx]}', fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Time (s)')
        axes[i].set_ylabel('Normalized Acc')
        axes[i].legend(fontsize=8)
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(processed_dir / 'sample_visualization_6class.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"   Saved: {processed_dir / 'sample_visualization_6class.png'}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✅ DATA READY FOR TRAINING (6 CLASSES)")
print("="*80)

summary = f"""
Dataset Summary:
  - Total samples: {len(y_labels):,}
  - Classes: 6 (0-5)
  - Features: {X_data.shape}
  - Labels: {y_labels.shape}
  
Class Distribution:
"""

for cls_idx in sorted(class_counts.keys()):
    count = class_counts[cls_idx]
    pct = count / len(y_labels) * 100
    summary += f"  {cls_idx}. {reverse_label_map[cls_idx]:30s}: {count:5d} ({pct:5.2f}%)\n"

summary += f"""
Imbalance: {max_count/min_count:.2f}x (max/min)

Quality Checks:
  - Fall_Initiation variance rank: #{fall_init_rank} {'✅' if fall_init_rank <= 2 else '⚠️'}
  - Data integrity: ✅ No NaN/Inf
  - Shape consistency: ✅ All aligned

Files Saved:
  - {processed_dir / 'X_data_6class.npy'}
  - {processed_dir / 'y_labels_6class.npy'}
  - {processed_dir / 'label_map_6class.json'}
  - {processed_dir / 'sample_visualization_6class.png'}

Next Steps:
  1. Train Micro-CNN: Run FallNet training notebook with MODEL_TYPE='micro_cnn'
  2. Expected accuracy: 85-90%
  3. Model size: ~41K params, Arduino-compatible
"""

print(summary)

# Save summary
with open(processed_dir / 'data_summary_6class.txt', 'w') as f:
    f.write(summary)

print(f"✅ Summary saved: {processed_dir / 'data_summary_6class.txt'}")

# %%
"""

---

## ✅ **Key Changes Made**

1. **Portable paths** - Works from any directory
2. **Saves 6-class data** with `_6class` suffix:
   - `X_data_6class.npy`
   - `y_labels_6class.npy`
   - `label_map_6class.json`
3. **Saves visualization** - `sample_visualization_6class.png`
4. **Saves text summary** - `data_summary_6class.txt`
5. **Better diagnostics** - Fall_Initiation variance rank check

---

## 🎯 **What You Get**

After running this notebook:
```
fall_detection_data/processed/
├── X_data.npy                      # Original 8-class
├── y_labels.npy                    # Original 8-class
├── X_data_6class.npy              # ✅ New 6-class
├── y_labels_6class.npy            # ✅ New 6-class
├── label_map_6class.json          # ✅ New mapping
├── sample_visualization_6class.png # ✅ Plots
└── data_summary_6class.txt        # ✅ Summary
"""

In [ ]:
# %% [markdown]
# # FallNet Training Pipeline
# CNN-LSTM ensemble + Micro-CNN for fall detection with 6 classes

# %%
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

try:
    from keras_lmu import LMU
    LMU_AVAILABLE = True
except ImportError:
    LMU_AVAILABLE = False
    print("⚠️  keras-lmu not available, LMU models disabled")

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# %% [markdown]
## 1. FallNet Model Architecture

# %%
class FallNet:
    """
    FallNet: Multiple architectures for Pre-Impact Fall Detection
    
    Available models:
    - CNN-only (original, 13.6M params)
    - Micro-CNN (Arduino-ready, 41K params)
    - LSTM-only
    - LMU-only (requires keras-lmu)
    - CNN-LSTM ensemble
    - CNN-LMU ensemble
    """
    
    def __init__(self, input_shape=(200, 6), n_classes=6):
        """
        Args:
            input_shape: (timesteps, features) = (200, 6)
            n_classes: Number of output classes (6)
        """
        self.input_shape = input_shape
        self.n_classes = n_classes
        self.model = None
    
    # ========================================================================
    # MICRO-CNN BRANCH (New! Arduino-compatible)
    # ========================================================================
    def build_micro_cnn(self):
        """
        Micro-CNN for Arduino Deployment
        
        Parameters: ~41K (vs 13.6M for original CNN)
        Size: ~164KB FP32, ~41KB INT8
        Target accuracy: 85-88%
        Arduino compatible: ✅ YES
        """
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        # Block 1: 32 filters
        x = layers.Conv1D(32, 3, padding='same', name='micro_conv1')(inputs)
        x = layers.BatchNormalization(name='micro_bn1')(x)
        x = layers.ReLU(name='micro_relu1')(x)
        x = layers.MaxPooling1D(2, name='micro_pool1')(x)  # (100, 32)
        
        # Block 2: 64 filters
        x = layers.Conv1D(64, 3, padding='same', name='micro_conv2')(x)
        x = layers.BatchNormalization(name='micro_bn2')(x)
        x = layers.ReLU(name='micro_relu2')(x)
        x = layers.MaxPooling1D(2, name='micro_pool2')(x)  # (50, 64)
        
        # Block 3: 128 filters
        x = layers.Conv1D(128, 3, padding='same', name='micro_conv3')(x)
        x = layers.BatchNormalization(name='micro_bn3')(x)
        x = layers.ReLU(name='micro_relu3')(x)
        x = layers.MaxPooling1D(2, name='micro_pool3')(x)  # (25, 128)
        
        # KEY: GlobalAveragePooling instead of Flatten + Dense(1024)
        x = layers.GlobalAveragePooling1D(name='micro_global_pool')(x)  # (128,)
        
        # Small classifier
        x = layers.Dense(64, activation='relu', name='micro_fc1')(x)
        x = layers.Dropout(0.3, name='micro_dropout')(x)
        outputs = layers.Dense(self.n_classes, activation='softmax', name='micro_output')(x)
        
        self.model = models.Model(
            inputs=inputs,
            outputs=outputs,
            name='FallNet_MicroCNN'
        )
        return self.model
    
    # ========================================================================
    # LMU BRANCH
    # ========================================================================
    def build_lmu_branch(self, inputs):
        """LMU Branch with regularization"""
        if not LMU_AVAILABLE:
            raise ImportError("keras-lmu not installed. Install with: uv add keras-lmu")
        
        x = LMU(
            memory_d=32,
            order=32,
            theta=200.0,
            hidden_cell=None,
            kernel_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),
            recurrent_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),
            dropout=0.3,
            recurrent_dropout=0.2,
            name='lmu'
        )(inputs)
        
        x = layers.Dense(
            128,
            activation='relu',
            kernel_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),
            name='lmu_dense1'
        )(x)
        x = layers.Dropout(0.4, name='lmu_dropout1')(x)
        
        x = layers.Dense(
            64,
            activation='relu',
            kernel_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),
            name='lmu_dense2'
        )(x)
        x = layers.Dropout(0.3, name='lmu_dropout2')(x)
        
        x = layers.Dense(
            32,
            activation='relu',
            kernel_regularizer=keras.regularizers.L1L2(l1=5e-4, l2=5e-4),
            name='lmu_dense3'
        )(x)
        x = layers.Dropout(0.2, name='lmu_dropout3')(x)
        
        lmu_output = layers.Dense(
            self.n_classes,
            activation='softmax',
            name='lmu_output'
        )(x)
        
        return lmu_output
    
    # ========================================================================
    # LSTM BRANCH
    # ========================================================================
    def build_lstm_branch(self, inputs):
        """LSTM Branch"""
        x = layers.LSTM(
            units=256,
            activation='tanh',
            return_sequences=False,
            name='lstm_layer'
        )(inputs)
        
        x = layers.Dense(128, activation='relu', name='lstm_dense1')(x)
        x = layers.Dropout(0.2, name='lstm_dropout1')(x)
        
        x = layers.Dense(64, activation='relu', name='lstm_dense2')(x)
        x = layers.Dropout(0.2, name='lstm_dropout2')(x)
        
        x = layers.Dense(32, activation='relu', name='lstm_dense3')(x)
        x = layers.Dropout(0.2, name='lstm_dropout3')(x)
        
        lstm_output = layers.Dense(
            self.n_classes,
            activation='softmax',
            name='lstm_output'
        )(x)
        
        return lstm_output
    
    # ========================================================================
    # CNN BRANCH (Original)
    # ========================================================================
    def build_cnn_branch(self, inputs):
        """CNN Branch (Original - 13.6M params)"""
        x = layers.Conv1D(
            filters=128,
            kernel_size=3,
            activation='relu',
            padding='same',
            name='conv1d_layer'
        )(inputs)
        
        x = layers.MaxPooling1D(pool_size=2, name='maxpool_layer')(x)
        x = layers.Flatten(name='flatten_layer')(x)
        
        x = layers.Dense(1024, activation='relu', name='cnn_dense1')(x)
        x = layers.Dropout(0.2, name='cnn_dropout1')(x)
        
        x = layers.Dense(512, activation='relu', name='cnn_dense2')(x)
        x = layers.Dropout(0.2, name='cnn_dropout2')(x)
        
        cnn_output = layers.Dense(
            self.n_classes,
            activation='softmax',
            name='cnn_output'
        )(x)
        
        return cnn_output
    
    # ========================================================================
    # STANDALONE MODELS
    # ========================================================================
    def build_cnn_only(self):
        """Build CNN-only model (Original - 13.6M params)"""
        inputs = layers.Input(shape=self.input_shape, name='input')
        cnn_output = self.build_cnn_branch(inputs)
        
        self.model = models.Model(
            inputs=inputs,
            outputs=cnn_output,
            name='FallNet_CNN_Only'
        )
        return self.model
    
    def build_lstm_only(self):
        """Build LSTM-only model"""
        inputs = layers.Input(shape=self.input_shape, name='input')
        lstm_output = self.build_lstm_branch(inputs)
        
        self.model = models.Model(
            inputs=inputs,
            outputs=lstm_output,
            name='FallNet_LSTM_Only'
        )
        return self.model
    
    def build_lmu_only(self):
        """Build LMU-only model"""
        inputs = layers.Input(shape=self.input_shape, name='input')
        lmu_output = self.build_lmu_branch(inputs)
        
        self.model = models.Model(
            inputs=inputs,
            outputs=lmu_output,
            name='FallNet_LMU_Only'
        )
        return self.model
    
    # ========================================================================
    # ENSEMBLE MODELS
    # ========================================================================
    def build_cnn_lstm_ensemble(self):
        """Build CNN-LSTM ensemble"""
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        lstm_output = self.build_lstm_branch(inputs)
        cnn_output = self.build_cnn_branch(inputs)
        
        ensemble_output = layers.Average(name='ensemble_average')([lstm_output, cnn_output])
        
        self.model = models.Model(
            inputs=inputs,
            outputs=ensemble_output,
            name='FallNet_CNN_LSTM'
        )
        
        return self.model
    
    def build_cnn_lmu_ensemble(self):
        """Build CNN-LMU ensemble"""
        inputs = layers.Input(shape=self.input_shape, name='input')
        
        lmu_output = self.build_lmu_branch(inputs)
        cnn_output = self.build_cnn_branch(inputs)
        
        ensemble_output = layers.Average(name='ensemble_average')([lmu_output, cnn_output])
        
        self.model = models.Model(
            inputs=inputs,
            outputs=ensemble_output,
            name='FallNet_CNN_LMU'
        )
        
        return self.model
    
    # ========================================================================
    # COMPILATION
    # ========================================================================
    def compile_model(self, learning_rate=1e-5):
        """Compile model with optimizer and metrics"""
        if self.model is None:
            raise ValueError("Model not built yet. Call build_*() first.")
        
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss='categorical_crossentropy',
            metrics=[
                'accuracy',
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall')
            ]
        )
        
        return self.model
    
    # ========================================================================
    # MODEL INFO
    # ========================================================================
    def get_model_info(self):
        """Get model parameter count and size estimates"""
        if self.model is None:
            return None
        
        total_params = self.model.count_params()
        trainable_params = sum([tf.size(w).numpy() for w in self.model.trainable_weights])
        
        return {
            'name': self.model.name,
            'total_params': total_params,
            'trainable_params': trainable_params,
            'fp32_size_mb': total_params * 4 / (1024**2),
            'int8_size_kb': total_params / 1024,
            'arduino_compatible': total_params < 100_000
        }


print("✅ FallNet class defined with Micro-CNN support")

# %% [markdown]
## 2. Model Selection and Building

# %%
print("\n" + "="*80)
print("AVAILABLE MODELS")
print("="*80)
print("""
Standalone Models:
  1. cnn_only        - Original CNN (13.6M params, 89.98% accuracy)
  2. micro_cnn       - Micro-CNN (41K params, ~87% accuracy, ✅ Arduino)
  3. lstm_only       - LSTM only (temporal encoding)
  4. lmu_only        - LMU only (requires keras-lmu)

Ensemble Models:
  5. cnn_lstm        - CNN-LSTM ensemble (14M params)
  6. cnn_lmu         - CNN-LMU ensemble (requires keras-lmu)
""")

# ========================================================================
# SELECT MODEL HERE
# ========================================================================
MODEL_TYPE = 'micro_cnn'  # ← CHANGE THIS TO SELECT MODEL
# Options: 'cnn_only', 'micro_cnn', 'lstm_only', 'lmu_only', 
#          'cnn_lstm', 'cnn_lmu'

print(f"\n🎯 Selected model: {MODEL_TYPE}")
print("="*80)

# Create instance
fallnet = FallNet(input_shape=(200, 6), n_classes=6)

# Build selected model
if MODEL_TYPE == 'cnn_only':
    model = fallnet.build_cnn_only()
elif MODEL_TYPE == 'micro_cnn':
    model = fallnet.build_micro_cnn()
elif MODEL_TYPE == 'lstm_only':
    model = fallnet.build_lstm_only()
elif MODEL_TYPE == 'lmu_only':
    model = fallnet.build_lmu_only()
elif MODEL_TYPE == 'cnn_lstm':
    model = fallnet.build_cnn_lstm_ensemble()
elif MODEL_TYPE == 'cnn_lmu':
    model = fallnet.build_cnn_lmu_ensemble()
else:
    raise ValueError(f"Unknown model type: {MODEL_TYPE}")

# Compile
model = fallnet.compile_model(learning_rate=1e-3)  # Higher LR for Micro-CNN

# Display architecture
print("\n📊 Model Architecture:")
model.summary()

# Get model info
info = fallnet.get_model_info()
print("\n" + "="*80)
print("MODEL SPECIFICATIONS")
print("="*80)
print(f"Name:             {info['name']}")
print(f"Total params:     {info['total_params']:,}")
print(f"Trainable params: {info['trainable_params']:,}")
print(f"FP32 size:        {info['fp32_size_mb']:.2f} MB")
print(f"INT8 size:        {info['int8_size_kb']:.2f} KB")
print(f"Arduino ready:    {'✅ YES' if info['arduino_compatible'] else '❌ NO'}")
print("="*80)

# %% [markdown]
## 3. Training Configuration

# %%
# Adjust batch size based on model
if MODEL_TYPE == 'micro_cnn':
    BATCH_SIZE = 64  # Can use larger batch for smaller model
    EPOCHS = 150
    PATIENCE = 20
else:
    BATCH_SIZE = 32
    EPOCHS = 200
    PATIENCE = 15

K_FOLDS = 5

print("\n" + "="*80)
print("TRAINING CONFIGURATION")
print("="*80)
print(f"Model:      {MODEL_TYPE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {EPOCHS}")
print(f"Patience:   {PATIENCE}")
print(f"K-Folds:    {K_FOLDS}")
print(f"Samples:    {len(y_labels):,}")

# %% [markdown]
## 4. Calculate Class Weights

# %%
print("\n" + "="*80)
print("CALCULATING CLASS WEIGHTS")
print("="*80)

# Calculate balanced weights
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_labels),
    y=y_labels
)

# Cap at 3x to prevent training instability
MAX_WEIGHT = 3.0
class_weights_array_capped = np.clip(class_weights_array, None, MAX_WEIGHT)
class_weights = dict(enumerate(class_weights_array_capped))

print("\nClass Distribution:")
counts = Counter(y_labels)
for cls_idx in range(6):
    count = counts[cls_idx]
    pct = count / len(y_labels) * 100
    weight = class_weights[cls_idx]
    print(f"  {reverse_label_map[cls_idx]:<30s}: {count:>5d} ({pct:>5.2f}%) → weight: {weight:.2f}x")

print(f"\n✅ Weight range: {min(class_weights.values()):.2f}x to {max(class_weights.values()):.2f}x")

# %% [markdown]
## 5. Verify Data Before Training

# %%
print("\n" + "="*80)
print("PRE-TRAINING VERIFICATION")
print("="*80)

print(f"✅ Data shapes:")
print(f"   X_data:        {X_data.shape}")
print(f"   y_labels:      {y_labels.shape}")
print(f"   y_categorical: {y_categorical.shape}")
print(f"\n✅ Classes: {len(np.unique(y_labels))} (should be 6)")
print(f"✅ Label range: {y_labels.min()}-{y_labels.max()} (should be 0-5)")
print(f"✅ Model output: {model.output_shape[-1]} (should be 6)")

assert X_data.shape[0] == y_labels.shape[0] == y_categorical.shape[0], "Shape mismatch!"
assert len(np.unique(y_labels)) == 6, "Should have 6 classes!"
assert y_labels.max() == 5, "Max label should be 5!"
assert model.output_shape[-1] == 6, "Model should output 6 classes!"

print("\n✅ All checks passed - ready to train!")

# %% [markdown]
## 6. K-Fold Cross-Validation Training

# %%
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_results = []
fold_histories = []

print("\n" + "="*80)
print(f"STARTING K-FOLD CROSS-VALIDATION - {MODEL_TYPE.upper()}")
print("="*80)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"{'='*80}")
    
    # Split data
    X_train, X_val = X_data[train_idx], X_data[val_idx]
    y_train, y_val = y_categorical[train_idx], y_categorical[val_idx]
    
    print(f"Train: {X_train.shape[0]:,} samples | Val: {X_val.shape[0]:,} samples")
    
    # Build fresh model for this fold
    fallnet_fold = FallNet(input_shape=(200, 6), n_classes=6)
    
    # Build same model type
    if MODEL_TYPE == 'cnn_only':
        model_fold = fallnet_fold.build_cnn_only()
    elif MODEL_TYPE == 'micro_cnn':
        model_fold = fallnet_fold.build_micro_cnn()
    elif MODEL_TYPE == 'lstm_only':
        model_fold = fallnet_fold.build_lstm_only()
    elif MODEL_TYPE == 'lmu_only':
        model_fold = fallnet_fold.build_lmu_only()
    elif MODEL_TYPE == 'cnn_lstm':
        model_fold = fallnet_fold.build_cnn_lstm_ensemble()
    elif MODEL_TYPE == 'cnn_lmu':
        model_fold = fallnet_fold.build_cnn_lmu_ensemble()
    
    model_fold = fallnet_fold.compile_model(learning_rate=1e-3 if MODEL_TYPE == 'micro_cnn' else 1e-5)
    
    # Define callbacks for THIS fold
    fold_callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=PATIENCE//2,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=str(output_dir / f'{MODEL_TYPE}_fold_{fold}.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
    ]
    
    # Train
    print(f"\nTraining fold {fold}...")
    history = model_fold.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=fold_callbacks,
        verbose=0
    )
    
    # Evaluate
    val_loss, val_acc, val_precision, val_recall = model_fold.evaluate(
        X_val, y_val, batch_size=BATCH_SIZE, verbose=0
    )
    val_f1 = 2 * (val_precision * val_recall) / (val_precision + val_recall) if (val_precision + val_recall) > 0 else 0
    
    print(f"\n{'='*50}")
    print(f"Fold {fold} Results:")
    print(f"{'='*50}")
    print(f"Loss:      {val_loss:.4f}")
    print(f"Accuracy:  {val_acc:.4f}")
    print(f"Precision: {val_precision:.4f}")
    print(f"Recall:    {val_recall:.4f}")
    print(f"F1-Score:  {val_f1:.4f}")
    
    # Store results
    fold_results.append({
        'fold': fold,
        'val_loss': val_loss,
        'val_accuracy': val_acc,
        'val_precision': val_precision,
        'val_recall': val_recall,
        'val_f1': val_f1
    })
    
    fold_histories.append(history.history)
    
    print(f"✅ Model saved: {MODEL_TYPE}_fold_{fold}.keras")

print("\n" + "="*80)
print("K-FOLD CROSS-VALIDATION COMPLETE")
print("="*80)

# %% [markdown]
## 7. Aggregate Results

# %%
results_df = pd.DataFrame(fold_results)

print("\n" + "="*80)
print(f"RESULTS ACROSS ALL FOLDS - {MODEL_TYPE.upper()}")
print("="*80)
print(results_df.to_string(index=False))

print("\n" + "="*80)
print("AVERAGE PERFORMANCE ± STD")
print("="*80)

mean_results = results_df.mean(numeric_only=True)
std_results = results_df.std(numeric_only=True)

metrics_table = []
for metric in ['val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1']:
    metrics_table.append({
        'Metric': metric,
        'Mean': f"{mean_results[metric]:.4f}",
        'Std': f"±{std_results[metric]:.4f}"
    })

metrics_df = pd.DataFrame(metrics_table)
print(metrics_df.to_string(index=False))

# Compare with baseline
print("\n" + "="*80)
print("COMPARISON WITH BASELINE")
print("="*80)
print(f"                    CNN-only (Baseline)    {MODEL_TYPE.upper()}")
print(f"Parameters:         13,639,992              {info['total_params']:,}")
print(f"Size (FP32):        54.5 MB                 {info['fp32_size_mb']:.2f} MB")
print(f"Size (INT8):        13.6 MB                 {info['int8_size_kb']:.2f} KB")
print(f"Accuracy:           89.98%                  {mean_results['val_accuracy']*100:.2f}%")
print(f"Arduino ready:      ❌ NO                    {'✅ YES' if info['arduino_compatible'] else '❌ NO'}")

# %% [markdown]
## 8-11. [Rest of the pipeline stays the same - visualization, confusion matrix, etc.]

# %%
# ... (keep all remaining cells unchanged)


In [ ]:
# %% [markdown]
# # Micro-CNN INT8 Quantization
# Quantize models to INT8 for Arduino deployment and measure accuracy degradation

# %%
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import json
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm.notebook import tqdm

print("="*80)
print("MICRO-CNN INT8 QUANTIZATION")
print("="*80)
print(f"TensorFlow version: {tf.__version__}")

# %% [markdown]
## 1. Setup Paths and Load Data

# %%
# Auto-detect project root
current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:
        project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
        break

# Define paths
data_dir = project_root / "fall_detection_data"
processed_dir = data_dir / "processed"
models_dir = data_dir / "models"
quantized_dir = models_dir / "quantized"
quantized_dir.mkdir(exist_ok=True)

print(f"📂 Project root: {project_root}")
print(f"📂 Models directory: {models_dir}")
print(f"📂 Quantized output: {quantized_dir}")

# Load data
print("\n📂 Loading data...")
X_data = np.load(processed_dir / "X_data_6class.npy")
y_labels = np.load(processed_dir / "y_labels_6class.npy")

with open(processed_dir / "label_map_6class.json", 'r') as f:
    label_map = json.load(f)

reverse_label_map = {v: k for k, v in label_map.items()}

# Convert to float32 (required for TFLite)
X_data = X_data.astype(np.float32)

print(f"   X shape: {X_data.shape}")
print(f"   y shape: {y_labels.shape}")
print(f"   Classes: {len(label_map)}")
print(f"   Data type: {X_data.dtype}")

# %% [markdown]
## 2. Define Quantization Functions

# %%
def representative_dataset_gen(X_subset):
    """
    Generator for representative dataset (calibration data)
    Used by TFLite converter to determine quantization parameters
    """
    for sample in X_subset:
        # TFLite expects batch dimension
        yield [np.expand_dims(sample, axis=0)]


def quantize_model(model_path, X_calibration, model_name):
    """
    Quantize a Keras model to INT8 TFLite format
    
    Args:
        model_path: Path to .keras model
        X_calibration: Calibration data for quantization
        model_name: Name for output file
    
    Returns:
        tflite_model: Quantized TFLite model bytes
        tflite_path: Path to saved .tflite file
    """
    print(f"\n{'='*60}")
    print(f"Quantizing: {model_path.name}")
    print(f"{'='*60}")
    
    # Load Keras model
    print("  Loading Keras model...")
    model = keras.models.load_model(model_path)
    
    # Get original model size
    original_size = model_path.stat().st_size
    param_count = model.count_params()
    
    print(f"  Original model:")
    print(f"    Parameters: {param_count:,}")
    print(f"    File size:  {original_size / (1024**2):.2f} MB ({original_size / 1024:.1f} KB)")
    
    # Convert to TFLite with INT8 quantization
    print("\n  Converting to TFLite INT8...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    # Enable INT8 quantization
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    
    # Provide representative dataset for full integer quantization
    converter.representative_dataset = lambda: representative_dataset_gen(X_calibration)
    
    # Ensure all ops are INT8
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    
    # Convert
    try:
        tflite_model = converter.convert()
    except Exception as e:
        print(f"  ⚠️  Strict INT8 failed: {e}")
        print(f"  Trying with float fallback...")
        
        # Fallback: allow float fallback for unsupported ops
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
            tf.lite.OpsSet.TFLITE_BUILTINS
        ]
        tflite_model = converter.convert()
    
    # Save TFLite model
    tflite_path = quantized_dir / f"{model_name}.tflite"
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    
    quantized_size = len(tflite_model)
    compression_ratio = original_size / quantized_size
    
    print(f"\n  ✅ Quantization complete!")
    print(f"    Quantized size: {quantized_size / 1024:.1f} KB")
    print(f"    Compression:    {compression_ratio:.1f}x smaller")
    print(f"    Saved to:       {tflite_path.name}")
    
    return tflite_model, tflite_path


def evaluate_tflite_model(tflite_path, X_test, y_test, show_progress=True):
    """
    Evaluate quantized TFLite model
    
    Args:
        tflite_path: Path to .tflite file
        X_test: Test data
        y_test: Test labels
        show_progress: Show progress bar
    
    Returns:
        accuracy: Test accuracy
        predictions: Model predictions
    """
    print(f"\n{'='*60}")
    print(f"Evaluating: {tflite_path.name}")
    print(f"{'='*60}")
    
    # Load TFLite model
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    
    # Get input/output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    print(f"  Input shape:  {input_details[0]['shape']}")
    print(f"  Input type:   {input_details[0]['dtype']}")
    print(f"  Output shape: {output_details[0]['shape']}")
    print(f"  Output type:  {output_details[0]['dtype']}")
    
    # Get quantization parameters
    input_scale = input_details[0]['quantization_parameters']['scales'][0]
    input_zero_point = input_details[0]['quantization_parameters']['zero_points'][0]
    output_scale = output_details[0]['quantization_parameters']['scales'][0]
    output_zero_point = output_details[0]['quantization_parameters']['zero_points'][0]
    
    print(f"\n  Quantization parameters:")
    print(f"    Input:  scale={input_scale:.6f}, zero={input_zero_point}")
    print(f"    Output: scale={output_scale:.6f}, zero={output_zero_point}")
    
    # Run inference
    print(f"\n  Running inference on {len(X_test):,} samples...")
    predictions = []
    
    iterator = tqdm(X_test, desc="  Progress") if show_progress else X_test
    
    for sample in iterator:
        # Quantize input
        input_data = sample / input_scale + input_zero_point
        input_data = input_data.astype(input_details[0]['dtype'])
        input_data = np.expand_dims(input_data, axis=0)
        
        # Set input tensor
        interpreter.set_tensor(input_details[0]['index'], input_data)
        
        # Run inference
        interpreter.invoke()
        
        # Get output tensor
        output_data = interpreter.get_tensor(output_details[0]['index'])
        
        # Dequantize output
        output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale
        
        # Get prediction
        pred = np.argmax(output_data[0])
        predictions.append(pred)
    
    predictions = np.array(predictions)
    accuracy = accuracy_score(y_test, predictions)
    
    print(f"\n  ✅ Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    return accuracy, predictions

print("✅ Functions defined")

# %% [markdown]
## 3. Quantize All Fold Models

# %%
print("\n" + "="*80)
print("QUANTIZING ALL FOLDS")
print("="*80)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    model_path = models_dir / f"micro_cnn_fold_{fold}.keras"
    
    if not model_path.exists():
        print(f"\n⚠️  Fold {fold}: Model not found, skipping...")
        continue
    
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/5")
    print(f"{'='*80}")
    
    # Get validation data for this fold
    X_val = X_data[val_idx]
    y_val = y_labels[val_idx]
    
    # Use subset of training data for calibration (representative dataset)
    X_train = X_data[train_idx]
    calibration_size = min(1000, len(X_train))  # Use 1000 samples for calibration
    np.random.seed(42)  # For reproducibility
    calibration_indices = np.random.choice(len(X_train), calibration_size, replace=False)
    X_calibration = X_train[calibration_indices]
    
    print(f"\nDataset sizes:")
    print(f"  Validation:  {len(X_val):,} samples")
    print(f"  Calibration: {len(X_calibration):,} samples (for quantization)")
    
    # Evaluate original Keras model (FP32)
    print(f"\n--- Original Keras Model (FP32) ---")
    model = keras.models.load_model(model_path)
    y_pred_keras = np.argmax(model.predict(X_val, verbose=0), axis=1)
    accuracy_keras = accuracy_score(y_val, y_pred_keras)
    print(f"  Accuracy: {accuracy_keras:.4f} ({accuracy_keras*100:.2f}%)")
    
    # Quantize to INT8
    tflite_model, tflite_path = quantize_model(
        model_path, 
        X_calibration,
        f"micro_cnn_fold_{fold}"
    )
    
    # Evaluate quantized model (INT8)
    print(f"\n--- Quantized TFLite Model (INT8) ---")
    accuracy_tflite, y_pred_tflite = evaluate_tflite_model(
        tflite_path,
        X_val,
        y_val,
        show_progress=True
    )
    
    # Calculate accuracy drop
    accuracy_drop = accuracy_keras - accuracy_tflite
    accuracy_drop_pct = (accuracy_drop / accuracy_keras) * 100
    
    # Calculate per-class metrics for quantized model
    fall_init_idx = label_map["Fall_Initiation"]
    fall_init_mask = y_val == fall_init_idx
    fall_init_recall_fp32 = (y_pred_keras[fall_init_mask] == fall_init_idx).mean()
    fall_init_recall_int8 = (y_pred_tflite[fall_init_mask] == fall_init_idx).mean()
    
    print(f"\n{'='*60}")
    print(f"FOLD {fold} SUMMARY")
    print(f"{'='*60}")
    print(f"  FP32 (Keras):       {accuracy_keras:.4f} ({accuracy_keras*100:.2f}%)")
    print(f"  INT8 (TFLite):      {accuracy_tflite:.4f} ({accuracy_tflite*100:.2f}%)")
    print(f"  Accuracy drop:      {accuracy_drop:.4f} ({accuracy_drop_pct:.2f}%)")
    print(f"\n  Fall_Initiation Recall:")
    print(f"    FP32:  {fall_init_recall_fp32:.4f} ({fall_init_recall_fp32*100:.2f}%)")
    print(f"    INT8:  {fall_init_recall_int8:.4f} ({fall_init_recall_int8*100:.2f}%)")
    print(f"    Drop:  {fall_init_recall_fp32 - fall_init_recall_int8:.4f}")
    
    # Store results
    results.append({
        'fold': fold,
        'accuracy_fp32': accuracy_keras,
        'accuracy_int8': accuracy_tflite,
        'accuracy_drop': accuracy_drop,
        'accuracy_drop_pct': accuracy_drop_pct,
        'fall_init_recall_fp32': fall_init_recall_fp32,
        'fall_init_recall_int8': fall_init_recall_int8,
        'y_val': y_val,
        'y_pred_fp32': y_pred_keras,
        'y_pred_int8': y_pred_tflite
    })

print("\n" + "="*80)
print("✅ ALL FOLDS QUANTIZED")
print("="*80)

# %% [markdown]
## 4. Aggregate Results

# %%
if results:
    results_df = pd.DataFrame([
        {
            'Fold': r['fold'],
            'FP32 Accuracy': f"{r['accuracy_fp32']:.4f}",
            'INT8 Accuracy': f"{r['accuracy_int8']:.4f}",
            'Drop (%)': f"{r['accuracy_drop_pct']:.2f}%",
            'Fall_Init FP32': f"{r['fall_init_recall_fp32']:.4f}",
            'Fall_Init INT8': f"{r['fall_init_recall_int8']:.4f}"
        }
        for r in results
    ])
    
    print("\n" + "="*80)
    print("RESULTS ACROSS ALL FOLDS")
    print("="*80)
    print()
    print(results_df.to_string(index=False))
    
    # Calculate statistics
    fp32_accs = [r['accuracy_fp32'] for r in results]
    int8_accs = [r['accuracy_int8'] for r in results]
    drops = [r['accuracy_drop'] for r in results]
    drop_pcts = [r['accuracy_drop_pct'] for r in results]
    fall_init_fp32 = [r['fall_init_recall_fp32'] for r in results]
    fall_init_int8 = [r['fall_init_recall_int8'] for r in results]
    
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    print(f"\nOverall Accuracy:")
    print(f"  FP32 (Keras):   {np.mean(fp32_accs):.4f} ± {np.std(fp32_accs):.4f} ({np.mean(fp32_accs)*100:.2f}%)")
    print(f"  INT8 (TFLite):  {np.mean(int8_accs):.4f} ± {np.std(int8_accs):.4f} ({np.mean(int8_accs)*100:.2f}%)")
    print(f"  Average drop:   {np.mean(drops):.4f} ({np.mean(drop_pcts):.2f}%)")
    print(f"  Max drop:       {np.max(drops):.4f} ({np.max(drop_pcts):.2f}%) - Fold {results[np.argmax(drops)]['fold']}")
    print(f"  Min drop:       {np.min(drops):.4f} ({np.min(drop_pcts):.2f}%) - Fold {results[np.argmin(drops)]['fold']}")
    
    print(f"\nFall_Initiation Recall (Critical):")
    print(f"  FP32:  {np.mean(fall_init_fp32):.4f} ± {np.std(fall_init_fp32):.4f} ({np.mean(fall_init_fp32)*100:.2f}%)")
    print(f"  INT8:  {np.mean(fall_init_int8):.4f} ± {np.std(fall_init_int8):.4f} ({np.mean(fall_init_int8)*100:.2f}%)")
    print(f"  Drop:  {np.mean(fall_init_fp32) - np.mean(fall_init_int8):.4f}")

else:
    print("\n❌ No results to analyze!")

# %% [markdown]
## 5. File Size Comparison

# %%
if results:
    print("\n" + "="*80)
    print("FILE SIZE COMPARISON")
    print("="*80)
    
    size_data = []
    
    for fold in range(1, 6):
        keras_path = models_dir / f"micro_cnn_fold_{fold}.keras"
        tflite_path = quantized_dir / f"micro_cnn_fold_{fold}.tflite"
        
        if keras_path.exists() and tflite_path.exists():
            keras_size = keras_path.stat().st_size
            tflite_size = tflite_path.stat().st_size
            ratio = keras_size / tflite_size
            
            size_data.append({
                'Fold': fold,
                'FP32 (.keras)': f"{keras_size / 1024:.1f} KB",
                'INT8 (.tflite)': f"{tflite_size / 1024:.1f} KB",
                'Compression': f"{ratio:.1f}x"
            })
    
    size_df = pd.DataFrame(size_data)
    print()
    print(size_df.to_string(index=False))
    
    # Average compression
    avg_keras = np.mean([float(d['FP32 (.keras)'].split()[0]) for d in size_data])
    avg_tflite = np.mean([float(d['INT8 (.tflite)'].split()[0]) for d in size_data])
    avg_compression = avg_keras / avg_tflite
    
    print(f"\nAverage sizes:")
    print(f"  FP32:        {avg_keras:.1f} KB")
    print(f"  INT8:        {avg_tflite:.1f} KB")
    print(f"  Compression: {avg_compression:.1f}x smaller")

# %% [markdown]
## 6. Arduino Compatibility Check

# %%
print("\n" + "="*80)
print("ARDUINO NANO 33 BLE SENSE COMPATIBILITY")
print("="*80)

tflite_path = quantized_dir / "micro_cnn_fold_1.tflite"
if tflite_path.exists():
    tflite_size = tflite_path.stat().st_size
    flash_available = 1024 * 1024  # 1 MB
    ram_available = 256 * 1024     # 256 KB
    estimated_ram = 80 * 1024      # Estimated
    
    print(f"\n📊 Flash Memory:")
    print(f"  Available:     {flash_available / 1024:.0f} KB (1 MB)")
    print(f"  Model size:    {tflite_size / 1024:.1f} KB")
    print(f"  Usage:         {(tflite_size / flash_available) * 100:.1f}%")
    print(f"  Remaining:     {(flash_available - tflite_size) / 1024:.0f} KB")
    print(f"  Status:        {'✅ FITS!' if tflite_size < flash_available else '❌ TOO BIG'}")
    
    print(f"\n📊 RAM (estimated):")
    print(f"  Available:     {ram_available / 1024:.0f} KB (256 KB)")
    print(f"  Model needs:   ~{estimated_ram / 1024:.0f} KB")
    print(f"  Usage:         ~{(estimated_ram / ram_available) * 100:.0f}%")
    print(f"  Remaining:     ~{(ram_available - estimated_ram) / 1024:.0f} KB")
    print(f"  Status:        {'✅ FITS!' if estimated_ram < ram_available else '❌ TOO BIG'}")
    
    print(f"\n✅ Arduino Deployment: READY!")
else:
    print("\n⚠️  No quantized model found for compatibility check")

# %% [markdown]
## 7. Visualization: Accuracy Comparison

# %%
if results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Accuracy comparison
    folds = [r['fold'] for r in results]
    fp32_vals = [r['accuracy_fp32'] * 100 for r in results]
    int8_vals = [r['accuracy_int8'] * 100 for r in results]
    
    x = np.arange(len(folds))
    width = 0.35
    
    axes[0].bar(x - width/2, fp32_vals, width, label='FP32 (Keras)', alpha=0.8, color='#2ecc71')
    axes[0].bar(x + width/2, int8_vals, width, label='INT8 (TFLite)', alpha=0.8, color='#3498db')
    
    axes[0].set_xlabel('Fold', fontsize=12)
    axes[0].set_ylabel('Accuracy (%)', fontsize=12)
    axes[0].set_title('FP32 vs INT8 Accuracy by Fold', fontsize=13, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(folds)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_ylim([92, 96])
    
    # Plot 2: Accuracy drop
    drop_vals = [r['accuracy_drop_pct'] for r in results]
    
    bars = axes[1].bar(folds, drop_vals, alpha=0.8, color='#e74c3c')
    axes[1].axhline(y=np.mean(drop_vals), color='#c0392b', linestyle='--', 
                    label=f'Mean: {np.mean(drop_vals):.2f}%', linewidth=2)
    
    axes[1].set_xlabel('Fold', fontsize=12)
    axes[1].set_ylabel('Accuracy Drop (%)', fontsize=12)
    axes[1].set_title('Quantization Accuracy Drop by Fold', fontsize=13, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}%', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(quantized_dir / 'quantization_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Visualization saved: {quantized_dir / 'quantization_comparison.png'}")

# %% [markdown]
## 8. Confusion Matrix: Best Fold (INT8)

# %%
if results:
    # Find best fold by INT8 accuracy
    best_fold_idx = np.argmax(int8_accs)
    best_fold = results[best_fold_idx]
    
    print(f"\n{'='*80}")
    print(f"CONFUSION MATRIX - BEST FOLD (#{best_fold['fold']}) - INT8")
    print(f"{'='*80}")
    print(f"INT8 Accuracy: {best_fold['accuracy_int8']:.4f} ({best_fold['accuracy_int8']*100:.2f}%)")
    
    cm = confusion_matrix(best_fold['y_val'], best_fold['y_pred_int8'])
    
    # Plot
    plt.figure(figsize=(10, 8))
    class_names = [reverse_label_map[i] for i in range(6)]
    
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Count'}
    )
    plt.title(f'Confusion Matrix - INT8 Quantized (Fold {best_fold["fold"]})\nAccuracy: {best_fold["accuracy_int8"]:.2%}',
              fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    
    plt.savefig(quantized_dir / 'confusion_matrix_int8.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Confusion matrix saved: {quantized_dir / 'confusion_matrix_int8.png'}")

# %% [markdown]
## 9. Final Summary

# %%
if results:
    print("\n" + "="*80)
    print("QUANTIZATION COMPLETE - FINAL SUMMARY")
    print("="*80)
    
    summary = f"""
✅ Successfully quantized {len(results)} models to INT8

Accuracy Performance:
  FP32 (Keras):   {np.mean(fp32_accs):.4f} ± {np.std(fp32_accs):.4f} ({np.mean(fp32_accs)*100:.2f}%)
  INT8 (TFLite):  {np.mean(int8_accs):.4f} ± {np.std(int8_accs):.4f} ({np.mean(int8_accs)*100:.2f}%)
  Average drop:   {np.mean(drops):.4f} ({np.mean(drop_pcts):.2f}%)
  
Fall_Initiation Recall (Critical):
  FP32:  {np.mean(fall_init_fp32):.4f} ({np.mean(fall_init_fp32)*100:.2f}%)
  INT8:  {np.mean(fall_init_int8):.4f} ({np.mean(fall_init_int8)*100:.2f}%)
  Drop:  {np.mean(fall_init_fp32) - np.mean(fall_init_int8):.4f}

File Sizes:
  FP32 (.keras):  ~{avg_keras:.0f} KB
  INT8 (.tflite): ~{avg_tflite:.0f} KB
  Compression:    ~{avg_compression:.0f}x smaller

Arduino Nano 33 BLE Sense:
  Flash: {tflite_size / 1024:.1f} KB / 1024 KB ({(tflite_size / flash_available) * 100:.1f}% used)
  RAM:   ~80 KB / 256 KB (~31% used)
  Status: ✅ READY FOR DEPLOYMENT!

Saved Files:
  - Quantized models: {quantized_dir}/micro_cnn_fold_*.tflite
  - Comparison plot:  {quantized_dir}/quantization_comparison.png
  - Confusion matrix: {quantized_dir}/confusion_matrix_int8.png

Next Steps:
  1. ✅ Models quantized to INT8
  2. ✅ Accuracy verified ({np.mean(drop_pcts):.2f}% drop is excellent!)
  3. 🎯 Deploy to Arduino Nano 33 BLE Sense
  4. 🎯 Convert to SNN for additional power savings
"""
    
    print(summary)
    
    # Save summary
    with open(quantized_dir / 'quantization_summary.txt', 'w') as f:
        f.write(summary)
    
    print(f"✅ Summary saved to: {quantized_dir / 'quantization_summary.txt'}")
    
else:
    print("\n❌ No quantization results available")


In [ ]:
# %%
# Install snnTorch and dependencies

import snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF
from snntorch import utils

import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

print("✅ snnTorch installed")
print(f"   Version: {snn.__version__}")

In [ ]:
# %% [markdown]
## SNN Architecture Definition
# %% [markdown]
## SNN Architecture Definition (Fixed)

# %%
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF

class MicroCNN_SNN(nn.Module):
    """
    Spiking Neural Network version of Micro-CNN
    
    Architecture:
    - 3 Conv layers with Leaky-Integrate-and-Fire (LIF) neurons
    - GlobalAveragePooling
    - 2 Dense layers with LIF neurons
    - Output: 6 classes
    """
    
    def __init__(self, num_classes=6, num_steps=25, beta=0.9, threshold=1.0):
        """
        Args:
            num_classes: Number of output classes (6)
            num_steps: Number of time steps for SNN simulation
            beta: Membrane potential decay rate (higher = longer memory)
            threshold: Spike threshold
        """
        super().__init__()
        
        self.num_classes = num_classes
        self.num_steps = num_steps
        
        # Surrogate gradient for backprop through spikes
        spike_grad = surrogate.fast_sigmoid(slope=25)
        
        # ====================================================================
        # Block 1: Conv + LIF + Pool
        # ====================================================================
        self.conv1 = nn.Conv1d(6, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.lif1 = snn.Leaky(
            beta=beta, 
            spike_grad=spike_grad, 
            init_hidden=False,  # ← Changed to False
            threshold=threshold
        )
        self.pool1 = nn.MaxPool1d(2)
        
        # ====================================================================
        # Block 2: Conv + LIF + Pool
        # ====================================================================
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.lif2 = snn.Leaky(
            beta=beta,
            spike_grad=spike_grad,
            init_hidden=False,  # ← Changed to False
            threshold=threshold
        )
        self.pool2 = nn.MaxPool1d(2)
        
        # ====================================================================
        # Block 3: Conv + LIF + Pool
        # ====================================================================
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.lif3 = snn.Leaky(
            beta=beta,
            spike_grad=spike_grad,
            init_hidden=False,  # ← Changed to False
            threshold=threshold
        )
        self.pool3 = nn.MaxPool1d(2)
        
        # ====================================================================
        # Global Average Pooling (convert to fully connected)
        # ====================================================================
        # After 3 pools: 200 → 100 → 50 → 25
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # ====================================================================
        # Classifier: Dense + LIF
        # ====================================================================
        self.fc1 = nn.Linear(128, 64)
        self.lif4 = snn.Leaky(
            beta=beta,
            spike_grad=spike_grad,
            init_hidden=False,  # ← Changed to False
            threshold=threshold
        )
        
        self.fc2 = nn.Linear(64, num_classes)
        self.lif_out = snn.Leaky(
            beta=beta,
            spike_grad=spike_grad,
            init_hidden=False,  # ← Changed to False
            threshold=threshold,
            output=True  # Final layer
        )
    
    def forward(self, x):
        """
        Forward pass with temporal dynamics
        
        Args:
            x: Input tensor [batch, channels=6, time=200]
        
        Returns:
            spk_out: Output spikes [num_steps, batch, num_classes]
            mem_out: Output membrane potentials [num_steps, batch, num_classes]
        """
        batch_size = x.size(0)
        
        # Initialize membrane potentials manually
        # We need to initialize based on the actual tensor shapes after convolutions
        
        # Record output spikes over time
        spk_out_rec = []
        mem_out_rec = []
        
        # Initialize hidden states to None - they'll be created on first pass
        mem1 = None
        mem2 = None
        mem3 = None
        mem4 = None
        mem_out = None
        
        # Temporal processing
        for step in range(self.num_steps):
            # Block 1
            cur1 = self.bn1(self.conv1(x))
            spk1, mem1 = self.lif1(cur1, mem1)  # Now mem1 is managed properly
            spk1_pooled = self.pool1(spk1)
            
            # Block 2
            cur2 = self.bn2(self.conv2(spk1_pooled))
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2_pooled = self.pool2(spk2)
            
            # Block 3
            cur3 = self.bn3(self.conv3(spk2_pooled))
            spk3, mem3 = self.lif3(cur3, mem3)
            spk3_pooled = self.pool3(spk3)
            
            # Global Average Pooling
            spk3_gap = self.global_pool(spk3_pooled)  # [batch, 128, 1]
            spk3_flat = spk3_gap.squeeze(-1)           # [batch, 128]
            
            # FC1
            cur4 = self.fc1(spk3_flat)
            spk4, mem4 = self.lif4(cur4, mem4)
            
            # Output layer
            cur_out = self.fc2(spk4)
            spk_out_step, mem_out = self.lif_out(cur_out, mem_out)
            
            spk_out_rec.append(spk_out_step)
            mem_out_rec.append(mem_out)
        
        # Stack recordings: [num_steps, batch, num_classes]
        spk_out_rec = torch.stack(spk_out_rec)
        mem_out_rec = torch.stack(mem_out_rec)
        
        return spk_out_rec, mem_out_rec


# Test the architecture
print("="*80)
print("BUILDING SNN MODEL")
print("="*80)

model = MicroCNN_SNN(num_classes=6, num_steps=25, beta=0.9, threshold=1.0)

# Test forward pass
batch_size = 4
x_test = torch.randn(batch_size, 6, 200)  # [batch, channels, time]

print(f"\nInput shape: {x_test.shape}")

with torch.no_grad():
    spk_out, mem_out = model(x_test)

print(f"Output spikes shape: {spk_out.shape}")  # [num_steps, batch, classes]
print(f"Output membrane shape: {mem_out.shape}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*80}")
print("MODEL PARAMETERS")
print(f"{'='*80}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Size (FP32):          {total_params * 4 / 1024:.2f} KB")
print(f"{'='*80}")

print("\n✅ SNN architecture defined successfully!")

In [ ]:
# %% [markdown]
# # Micro-CNN to SNN Conversion
# Converting the Micro-CNN architecture to a Spiking Neural Network

# %%
# Imports
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF
import numpy as np

print("✅ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"snnTorch version: {snn.__version__}")

# %% [markdown]
## Define SNN Architecture

# %%
class MicroCNN_SNN(nn.Module):
    """
    Spiking Neural Network version of Micro-CNN
    """
    
    def __init__(self, num_classes=6, num_steps=25, beta=0.9, threshold=1.0):
        super().__init__()
        
        self.num_classes = num_classes
        self.num_steps = num_steps
        
        spike_grad = surrogate.fast_sigmoid(slope=25)
        
        # Block 1
        self.conv1 = nn.Conv1d(6, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, 
                              init_hidden=False, threshold=threshold)
        self.pool1 = nn.MaxPool1d(2)
        
        # Block 2
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, 
                              init_hidden=False, threshold=threshold)
        self.pool2 = nn.MaxPool1d(2)
        
        # Block 3
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad, 
                              init_hidden=False, threshold=threshold)
        self.pool3 = nn.MaxPool1d(2)
        
        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Classifier
        self.fc1 = nn.Linear(128, 64)
        self.lif4 = snn.Leaky(beta=beta, spike_grad=spike_grad, 
                              init_hidden=False, threshold=threshold)
        
        self.fc2 = nn.Linear(64, num_classes)
        self.lif_out = snn.Leaky(beta=beta, spike_grad=spike_grad, 
                                 init_hidden=False, threshold=threshold, output=True)
    
    def forward(self, x):
        batch_size = x.size(0)
        
        spk_out_rec = []
        mem_out_rec = []
        
        mem1 = None
        mem2 = None
        mem3 = None
        mem4 = None
        mem_out = None
        
        for step in range(self.num_steps):
            # Block 1
            cur1 = self.bn1(self.conv1(x))
            spk1, mem1 = self.lif1(cur1, mem1)
            spk1_pooled = self.pool1(spk1)
            
            # Block 2
            cur2 = self.bn2(self.conv2(spk1_pooled))
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2_pooled = self.pool2(spk2)
            
            # Block 3
            cur3 = self.bn3(self.conv3(spk2_pooled))
            spk3, mem3 = self.lif3(cur3, mem3)
            spk3_pooled = self.pool3(spk3)
            
            # Global pooling
            spk3_gap = self.global_pool(spk3_pooled)
            spk3_flat = spk3_gap.squeeze(-1)
            
            # FC layers
            cur4 = self.fc1(spk3_flat)
            spk4, mem4 = self.lif4(cur4, mem4)
            
            cur_out = self.fc2(spk4)
            spk_out_step, mem_out = self.lif_out(cur_out, mem_out)
            
            spk_out_rec.append(spk_out_step)
            mem_out_rec.append(mem_out)
        
        spk_out_rec = torch.stack(spk_out_rec)
        mem_out_rec = torch.stack(mem_out_rec)
        
        return spk_out_rec, mem_out_rec

print("✅ MicroCNN_SNN class defined")

# %% [markdown]
## Test the Model

# %%
print("="*80)
print("TESTING SNN MODEL")
print("="*80)

# Create model
model = MicroCNN_SNN(num_classes=6, num_steps=25, beta=0.9, threshold=1.0)

# Test input
x_test = torch.randn(4, 6, 200)  # [batch, channels, time]
print(f"\nInput shape: {x_test.shape}")

# Forward pass
with torch.no_grad():
    spk_out, mem_out = model(x_test)

print(f"\nOutput shapes:")
print(f"  Spikes:    {spk_out.shape}")  # [25, 4, 6]
print(f"  Membranes: {mem_out.shape}")  # [25, 4, 6]

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")
print(f"Model size:       {total_params * 4 / 1024:.2f} KB")

print("\n✅ SNN model working correctly!")

# %%

In [ ]:
# %% [markdown]
# # ANN-to-SNN Conversion (Weight Transfer)
# Convert trained Micro-CNN weights to SNN without retraining

# %%
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow import keras
import numpy as np
import snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF
from pathlib import Path
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("="*80)
print("ANN-TO-SNN WEIGHT CONVERSION")
print("="*80)

# Setup paths
current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:
        project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
        break

data_dir = project_root / "fall_detection_data"
processed_dir = data_dir / "processed"
models_dir = data_dir / "models"
snn_dir = models_dir / "snn"
snn_dir.mkdir(exist_ok=True)

print(f"\n📂 Paths configured")
print(f"   TensorFlow models: {models_dir}")
print(f"   Converted SNNs: {snn_dir}")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")

# %% [markdown]
## 1. Define SNN Architecture (Matching CNN)

# %%
class MicroCNN_SNN(nn.Module):
    """SNN with same architecture as trained CNN"""
    
    def __init__(self, num_classes=6, num_steps=100, threshold=1.0):
        super().__init__()
        
        self.num_classes = num_classes
        self.num_steps = num_steps
        
        spike_grad = surrogate.fast_sigmoid(slope=25)
        
        # Block 1: Conv + BN + LIF + Pool
        self.conv1 = nn.Conv1d(6, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.lif1 = snn.Leaky(beta=0.99, spike_grad=spike_grad, 
                              init_hidden=False, threshold=threshold)
        self.pool1 = nn.MaxPool1d(2)
        
        # Block 2: Conv + BN + LIF + Pool
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.lif2 = snn.Leaky(beta=0.99, spike_grad=spike_grad, 
                              init_hidden=False, threshold=threshold)
        self.pool2 = nn.MaxPool1d(2)
        
        # Block 3: Conv + BN + LIF + Pool
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.lif3 = snn.Leaky(beta=0.99, spike_grad=spike_grad, 
                              init_hidden=False, threshold=threshold)
        self.pool3 = nn.MaxPool1d(2)
        
        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Classifier
        self.fc1 = nn.Linear(128, 64)
        self.lif4 = snn.Leaky(beta=0.99, spike_grad=spike_grad, 
                              init_hidden=False, threshold=threshold)
        
        self.fc2 = nn.Linear(64, num_classes)
        self.lif_out = snn.Leaky(beta=0.99, spike_grad=spike_grad, 
                                 init_hidden=False, threshold=threshold, output=True)
    
    def forward(self, x):
        spk_out_rec = []
        mem_out_rec = []
        
        mem1 = None
        mem2 = None
        mem3 = None
        mem4 = None
        mem_out = None
        
        for step in range(self.num_steps):
            # Block 1
            cur1 = self.bn1(self.conv1(x))
            spk1, mem1 = self.lif1(cur1, mem1)
            spk1_pooled = self.pool1(spk1)
            
            # Block 2
            cur2 = self.bn2(self.conv2(spk1_pooled))
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2_pooled = self.pool2(spk2)
            
            # Block 3
            cur3 = self.bn3(self.conv3(spk2_pooled))
            spk3, mem3 = self.lif3(cur3, mem3)
            spk3_pooled = self.pool3(spk3)
            
            # Global pooling
            spk3_gap = self.global_pool(spk3_pooled)
            spk3_flat = spk3_gap.squeeze(-1)
            
            # FC layers
            cur4 = self.fc1(spk3_flat)
            spk4, mem4 = self.lif4(cur4, mem4)
            
            cur_out = self.fc2(spk4)
            spk_out_step, mem_out = self.lif_out(cur_out, mem_out)
            
            spk_out_rec.append(spk_out_step)
            mem_out_rec.append(mem_out)
        
        spk_out_rec = torch.stack(spk_out_rec)
        mem_out_rec = torch.stack(mem_out_rec)
        
        return spk_out_rec, mem_out_rec

print("✅ SNN architecture defined")

# %% [markdown]
## 2. Weight Conversion Function

# %%
def convert_keras_to_pytorch_snn(keras_model_path, num_steps=100, threshold=1.0):
    """
    Convert trained Keras CNN to PyTorch SNN
    
    Args:
        keras_model_path: Path to .keras model file
        num_steps: Number of SNN time steps
        threshold: Spike threshold
    
    Returns:
        snn_model: PyTorch SNN with converted weights
    """
    print(f"\n{'='*60}")
    print(f"Converting: {keras_model_path.name}")
    print(f"{'='*60}")
    
    # Load Keras model
    print("  Loading Keras model...")
    keras_model = keras.models.load_model(keras_model_path)
    
    # Create SNN
    print("  Creating SNN...")
    snn_model = MicroCNN_SNN(
        num_classes=6,
        num_steps=num_steps,
        threshold=threshold
    )
    
    # Weight mapping
    print("  Transferring weights...")
    
    # Conv1 + BN1
    snn_model.conv1.weight.data = torch.FloatTensor(
        keras_model.get_layer('micro_conv1').get_weights()[0].transpose(2, 1, 0)
    )
    snn_model.conv1.bias.data = torch.FloatTensor(
        keras_model.get_layer('micro_conv1').get_weights()[1]
    )
    
    bn1_weights = keras_model.get_layer('micro_bn1').get_weights()
    snn_model.bn1.weight.data = torch.FloatTensor(bn1_weights[0])  # gamma
    snn_model.bn1.bias.data = torch.FloatTensor(bn1_weights[1])    # beta
    snn_model.bn1.running_mean.data = torch.FloatTensor(bn1_weights[2])
    snn_model.bn1.running_var.data = torch.FloatTensor(bn1_weights[3])
    
    # Conv2 + BN2
    snn_model.conv2.weight.data = torch.FloatTensor(
        keras_model.get_layer('micro_conv2').get_weights()[0].transpose(2, 1, 0)
    )
    snn_model.conv2.bias.data = torch.FloatTensor(
        keras_model.get_layer('micro_conv2').get_weights()[1]
    )
    
    bn2_weights = keras_model.get_layer('micro_bn2').get_weights()
    snn_model.bn2.weight.data = torch.FloatTensor(bn2_weights[0])
    snn_model.bn2.bias.data = torch.FloatTensor(bn2_weights[1])
    snn_model.bn2.running_mean.data = torch.FloatTensor(bn2_weights[2])
    snn_model.bn2.running_var.data = torch.FloatTensor(bn2_weights[3])
    
    # Conv3 + BN3
    snn_model.conv3.weight.data = torch.FloatTensor(
        keras_model.get_layer('micro_conv3').get_weights()[0].transpose(2, 1, 0)
    )
    snn_model.conv3.bias.data = torch.FloatTensor(
        keras_model.get_layer('micro_conv3').get_weights()[1]
    )
    
    bn3_weights = keras_model.get_layer('micro_bn3').get_weights()
    snn_model.bn3.weight.data = torch.FloatTensor(bn3_weights[0])
    snn_model.bn3.bias.data = torch.FloatTensor(bn3_weights[1])
    snn_model.bn3.running_mean.data = torch.FloatTensor(bn3_weights[2])
    snn_model.bn3.running_var.data = torch.FloatTensor(bn3_weights[3])
    
    # FC1
    fc1_weights = keras_model.get_layer('micro_fc1').get_weights()
    snn_model.fc1.weight.data = torch.FloatTensor(fc1_weights[0].T)
    snn_model.fc1.bias.data = torch.FloatTensor(fc1_weights[1])
    
    # FC2 (output)
    fc2_weights = keras_model.get_layer('micro_output').get_weights()
    snn_model.fc2.weight.data = torch.FloatTensor(fc2_weights[0].T)
    snn_model.fc2.bias.data = torch.FloatTensor(fc2_weights[1])
    
    print("  ✅ Weight transfer complete!")
    
    return snn_model

print("✅ Conversion function defined")

# %% [markdown]
## 3. Load Data

# %%
# Load data
X_data = np.load(processed_dir / "X_data_6class.npy")
y_labels = np.load(processed_dir / "y_labels_6class.npy")

with open(processed_dir / "label_map_6class.json", 'r') as f:
    label_map = json.load(f)

reverse_label_map = {v: k for k, v in label_map.items()}

# Convert to PyTorch
X_data_torch = torch.FloatTensor(X_data).permute(0, 2, 1)
y_labels_torch = torch.LongTensor(y_labels)

print(f"✅ Data loaded: {X_data_torch.shape}")

# %% [markdown]
## 4. Convert All Fold Models

# %%
print("\n" + "="*80)
print("CONVERTING ALL FOLD MODELS")
print("="*80)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

# Test different time steps to find optimal
NUM_STEPS_OPTIONS = [50, 100, 200]
best_num_steps = None
best_overall_acc = 0

for num_steps in NUM_STEPS_OPTIONS:
    print(f"\n{'='*80}")
    print(f"TESTING WITH {num_steps} TIME STEPS")
    print(f"{'='*80}")
    
    fold_accs = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_data_torch, y_labels_torch), 1):
        keras_model_path = models_dir / f"micro_cnn_fold_{fold}.keras"
        
        if not keras_model_path.exists():
            print(f"  ⚠️  Fold {fold}: Model not found, skipping...")
            continue
        
        # Convert model
        snn_model = convert_keras_to_pytorch_snn(
            keras_model_path,
            num_steps=num_steps,
            threshold=1.0
        )
        snn_model = snn_model.to(device)
        snn_model.eval()
        
        # Get validation data
        X_val = X_data_torch[val_idx].to(device)
        y_val = y_labels_torch[val_idx]
        
        # Evaluate
        print(f"  Evaluating fold {fold}...")
        all_preds = []
        
        with torch.no_grad():
            # Process in batches
            batch_size = 32
            for i in range(0, len(X_val), batch_size):
                batch_x = X_val[i:i+batch_size]
                
                spk_out, mem_out = snn_model(batch_x)
                
                # Predictions: sum spikes over time
                _, predicted = spk_out.sum(dim=0).max(1)
                all_preds.extend(predicted.cpu().numpy())
        
        # Calculate accuracy
        accuracy = accuracy_score(y_val.numpy(), all_preds)
        fold_accs.append(accuracy)
        
        print(f"  Fold {fold}: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    mean_acc = np.mean(fold_accs)
    print(f"\n  Mean accuracy with {num_steps} steps: {mean_acc:.4f} ({mean_acc*100:.2f}%)")
    
    if mean_acc > best_overall_acc:
        best_overall_acc = mean_acc
        best_num_steps = num_steps

print(f"\n✅ Best number of time steps: {best_num_steps} ({best_overall_acc*100:.2f}%)")

# %% [markdown]
## 5. Final Evaluation with Best Time Steps

# %%
print("\n" + "="*80)
print(f"FINAL EVALUATION - {best_num_steps} TIME STEPS")
print("="*80)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data_torch, y_labels_torch), 1):
    keras_model_path = models_dir / f"micro_cnn_fold_{fold}.keras"
    
    if not keras_model_path.exists():
        continue
    
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/5")
    print(f"{'='*60}")
    
    # Convert
    snn_model = convert_keras_to_pytorch_snn(
        keras_model_path,
        num_steps=best_num_steps,
        threshold=1.0
    )
    snn_model = snn_model.to(device)
    snn_model.eval()
    
    # Save converted model
    torch.save(snn_model.state_dict(), snn_dir / f"snn_converted_fold_{fold}.pth")
    
    # Evaluate
    X_val = X_data_torch[val_idx].to(device)
    y_val = y_labels_torch[val_idx]
    
    all_preds = []
    with torch.no_grad():
        batch_size = 32
        for i in range(0, len(X_val), batch_size):
            batch_x = X_val[i:i+batch_size]
            spk_out, mem_out = snn_model(batch_x)
            _, predicted = spk_out.sum(dim=0).max(1)
            all_preds.extend(predicted.cpu().numpy())
    
    accuracy = accuracy_score(y_val.numpy(), all_preds)
    
    # Fall_Initiation recall
    fall_init_idx = label_map["Fall_Initiation"]
    fall_init_mask = y_val.numpy() == fall_init_idx
    fall_init_recall = (np.array(all_preds)[fall_init_mask] == fall_init_idx).mean()
    
    print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Fall_Initiation Recall: {fall_init_recall:.4f} ({fall_init_recall*100:.2f}%)")
    
    fold_results.append({
        'fold': fold,
        'val_accuracy': accuracy,
        'fall_init_recall': fall_init_recall,
        'val_preds': all_preds,
        'val_targets': y_val.numpy()
    })

# %% [markdown]
## 6. Results Summary & Comparison

# %%
print("\n" + "="*80)
print("CONVERTED SNN RESULTS")
print("="*80)

accuracies = [r['val_accuracy'] for r in fold_results]
fall_recalls = [r['fall_init_recall'] for r in fold_results]

results_df = pd.DataFrame([
    {
        'Fold': r['fold'],
        'Accuracy': f"{r['val_accuracy']:.4f}",
        'Fall_Init_Recall': f"{r['fall_init_recall']:.4f}"
    }
    for r in fold_results
])

print("\nResults Across Folds:")
print(results_df.to_string(index=False))

print(f"\n{'='*80}")
print("SUMMARY STATISTICS")
print(f"{'='*80}")
print(f"Accuracy:  {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f} ({np.mean(accuracies)*100:.2f}%)")
print(f"Fall_Init: {np.mean(fall_recalls):.4f} ± {np.std(fall_recalls):.4f} ({np.mean(fall_recalls)*100:.2f}%)")

# Comparison
print(f"\n{'='*80}")
print("COMPARISON: CNN vs CONVERTED SNN")
print(f"{'='*80}")
print(f"\n                        CNN (FP32)    SNN (Converted)")
print(f"{'-'*65}")
print(f"Accuracy:               94.71%        {np.mean(accuracies)*100:.2f}%")
print(f"Drop:                   -             {94.71 - np.mean(accuracies)*100:.2f}%")
print(f"Fall_Init Recall:       97.82%        {np.mean(fall_recalls)*100:.2f}%")
print(f"Time Steps:             1             {best_num_steps}")
print(f"Training Time:          Hours         Seconds! ✅")
print(f"{'-'*65}")

print(f"\n✅ Conversion complete! No training required!")
print(f"✅ Models saved to: {snn_dir}")

# %%

In [ ]:
# %% [markdown]
## SNN Model - Clean Implementation
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
torch.cuda.empty_cache()
# %%
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
import numpy as np

class MicroCNN_SNN(nn.Module):
    """Clean SNN implementation without manual detaching"""
    
    def __init__(self, num_classes=6, num_steps=25, beta=0.95, threshold=1.0):
        super().__init__()
        
        self.num_classes = num_classes
        self.num_steps = num_steps
        
        # Surrogate gradient for backprop
        spike_grad = surrogate.fast_sigmoid(slope=25)
        
        # Block 1: Conv -> BN -> LIF -> Pool
        self.conv1 = nn.Conv1d(6, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool1 = nn.MaxPool1d(2)
        
        # Block 2: Conv -> BN -> LIF -> Pool
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool2 = nn.MaxPool1d(2)
        
        # Block 3: Conv -> BN -> LIF -> Pool
        self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool3 = nn.MaxPool1d(2)
        
        # Global pooling
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128, 64)
        self.lif4 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        
        self.fc2 = nn.Linear(64, num_classes)
        self.lif_out = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
    
    def forward(self, x):
        """
        Args:
            x: [batch, channels, time_steps] input tensor
        
        Returns:
            spk_rec: [num_steps, batch, num_classes] spike recordings
            mem_rec: [num_steps, batch, num_classes] membrane recordings
        """
        batch_size = x.size(0)
        
        # Initialize membrane potentials
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()
        mem_out = self.lif_out.init_leaky()
        
        # Recording lists
        spk_rec = []
        mem_rec = []
        
        # Process over time steps
        for step in range(self.num_steps):
            # Block 1
            cur1 = self.bn1(self.conv1(x))
            spk1, mem1 = self.lif1(cur1, mem1)
            spk1 = self.pool1(spk1)
            
            # Block 2
            cur2 = self.bn2(self.conv2(spk1))
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2 = self.pool2(spk2)
            
            # Block 3
            cur3 = self.bn3(self.conv3(spk2))
            spk3, mem3 = self.lif3(cur3, mem3)
            spk3 = self.pool3(spk3)
            
            # Global average pooling
            spk3 = self.global_pool(spk3).squeeze(-1)
            
            # FC1
            cur4 = self.fc1(spk3)
            spk4, mem4 = self.lif4(cur4, mem4)
            
            # Output layer
            cur_out = self.fc2(spk4)
            spk_out, mem_out = self.lif_out(cur_out, mem_out)
            
            # Record
            spk_rec.append(spk_out)
            mem_rec.append(mem_out)
        
        # Stack: [num_steps, batch, num_classes]
        return torch.stack(spk_rec), torch.stack(mem_rec)


print("✅ SNN Model defined")

# Test forward pass
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_test = MicroCNN_SNN(6, 25, 0.95, 1.0).to(device)
x_test = torch.randn(4, 6, 200).to(device)

print("\nTesting forward pass...")
with torch.no_grad():
    spk, mem = model_test(x_test)
    print(f"✅ Success!")
    print(f"   Spikes: {spk.shape}")
    print(f"   Membrane: {mem.shape}")

# %% [markdown]
## Training Loop

# %%
# Configuration
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 5e-4
NUM_STEPS = 25
BETA = 0.95
THRESHOLD = 1.0

print("="*80)
print("SNN TRAINING")
print("="*80)
print(f"Batch: {BATCH_SIZE} | Epochs: {EPOCHS} | Steps: {NUM_STEPS}")
print(f"Device: {device}")

# Dataset
class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Assuming X_data_torch and y_labels_torch are already defined
dataset = FallDataset(X_data_torch, y_labels_torch)

# Single fold for testing
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(skf.split(X_data_torch, y_labels_torch))

train_loader = DataLoader(
    Subset(dataset, train_idx), 
    batch_size=BATCH_SIZE, 
    shuffle=True
)
val_loader = DataLoader(
    Subset(dataset, val_idx), 
    batch_size=BATCH_SIZE, 
    shuffle=False
)

print(f"\nTrain samples: {len(train_idx):,}")
print(f"Val samples: {len(val_idx):,}")

# Initialize model
model = MicroCNN_SNN(6, NUM_STEPS, BETA, THRESHOLD).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\n" + "="*80)
print("Starting Training...")
print("="*80)

# Training
best_acc = 0
history = {'train_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    # ========================
    # Training Phase
    # ========================
    model.train()
    epoch_loss = 0
    
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device)
        targets = targets.to(device)
        
        # Forward pass
        spk_out, mem_out = model(data)
        
        # Average membrane potential over time
        # [num_steps, batch, classes] -> [batch, classes]
        mem_avg = mem_out.mean(dim=0)
        
        # Compute loss
        loss = criterion(mem_avg, targets)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    
    # ========================
    # Validation Phase
    # ========================
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for data, targets in val_loader:
            data = data.to(device)
            targets = targets.to(device)
            
            # Forward pass
            spk_out, mem_out = model(data)
            
            # Prediction: sum spikes over time
            spk_sum = spk_out.sum(dim=0)  # [batch, classes]
            _, predicted = spk_sum.max(1)
            
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    
    val_acc = correct / total
    history['val_acc'].append(val_acc)
    
    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), snn_dir / 'snn_best.pth')
    
    # Print progress
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | Best: {best_acc:.4f}")

print("\n" + "="*80)
print("Training Complete!")
print("="*80)

# %% [markdown]
## Final Evaluation

# %%
# Load best model
model.load_state_dict(torch.load(snn_dir / 'snn_best.pth'))
model.eval()

all_preds = []
all_targets = []

print("Running final evaluation...")

with torch.no_grad():
    for data, targets in val_loader:
        data = data.to(device)
        targets = targets.to(device)
        
        spk_out, mem_out = model(data)
        spk_sum = spk_out.sum(dim=0)
        _, predicted = spk_sum.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

# Calculate metrics
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Fall_Initiation recall
fall_init_idx = label_map["Fall_Initiation"]
fall_init_mask = all_targets == fall_init_idx
fall_init_recall = (all_preds[fall_init_mask] == fall_init_idx).mean()

# ========================
# Results
# ========================
print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
print(f"Best Validation Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Fall_Initiation Recall:   {fall_init_recall:.4f} ({fall_init_recall*100:.2f}%)")

# ========================
# CNN vs SNN Comparison
# ========================
print("\n" + "="*80)
print("CNN vs SNN COMPARISON")
print("="*80)
print(f"{'Metric':<25} {'CNN (FP32)':<15} {'SNN (Trained)':<15} {'Drop':<10}")
print("-"*65)
print(f"{'Accuracy':<25} {94.71:<15.2f}% {best_acc*100:<15.2f}% {94.71 - best_acc*100:<10.2f}%")
print(f"{'Fall_Init Recall':<25} {97.82:<15.2f}% {fall_init_recall*100:<15.2f}% {97.82 - fall_init_recall*100:<10.2f}%")
print("-"*65)

# Performance assessment
accuracy_drop = 94.71 - best_acc*100
if accuracy_drop < 7:
    print("\n✅ EXCELLENT! Accuracy drop < 7%")
elif accuracy_drop < 10:
    print("\n✅ GOOD! Accuracy drop < 10%")
elif accuracy_drop < 15:
    print("\n⚠️  ACCEPTABLE! Accuracy drop < 15%")
else:
    print("\n❌ NEEDS TUNING! Accuracy drop > 15%")

# ========================
# Classification Report
# ========================
print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)
class_names = [reverse_label_map[i] for i in range(6)]
report = classification_report(all_targets, all_preds, target_names=class_names, digits=4)
print(report)

print(f"\n✅ Best model saved to: {snn_dir / 'snn_best.pth'}")

# %%

In [ ]:
X_data = np.load(processed_dir / "X_data.npy")
print(X_data.shape)  # is it [N, 200, 6] or [N, 6, 200]?

In [ ]:
print(processed_dir / "X_data.npy")
print(len(X_data), len(y_labels))
print(np.unique(y_labels, return_counts=True))

In [ ]:
# %% [markdown]
## Training Loop with 5-Fold Cross-Validation

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")
# Convert to PyTorch
X_data_torch = torch.FloatTensor(X_data).permute(0, 2, 1)
y_labels_torch = torch.LongTensor(y_labels)
snn_dir = models_dir / "snn"
snn_dir.mkdir(exist_ok=True)

print(f"✅ Data loaded: {X_data_torch.shape}")
# %%
# Configuration
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 5e-4
NUM_STEPS = 25
BETA = 0.95
THRESHOLD = 1.0
N_FOLDS = 5

print("="*80)
print("SNN TRAINING - 5-FOLD CROSS-VALIDATION")
print("="*80)
print(f"Batch: {BATCH_SIZE} | Epochs: {EPOCHS} | Steps: {NUM_STEPS}")
print(f"Folds: {N_FOLDS}")
print(f"Device: {device}")

# Dataset
class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Assuming X_data_torch and y_labels_torch are already defined
dataset = FallDataset(X_data_torch, y_labels_torch)

# 5-Fold Cross-Validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Store results for each fold
fold_results = {
    'val_acc': [],
    'fall_init_recall': [],
    'train_history': [],
    'predictions': [],
    'targets': []
}

print("\n" + "="*80)
print("Starting 5-Fold Cross-Validation...")
print("="*80)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data_torch, y_labels_torch), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{N_FOLDS}")
    print(f"{'='*80}")
    print(f"Train samples: {len(train_idx):,} | Val samples: {len(val_idx):,}")
    
    # Create data loaders
    train_loader = DataLoader(
        Subset(dataset, train_idx), 
        batch_size=BATCH_SIZE, 
        shuffle=True
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx), 
        batch_size=BATCH_SIZE, 
        shuffle=False
    )
    
    # Initialize fresh model for this fold
    model = MicroCNN_SNN(6, NUM_STEPS, BETA, THRESHOLD).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # Training history for this fold
    best_acc = 0
    fold_history = {'train_loss': [], 'val_acc': []}
    
    # ========================
    # Training Loop
    # ========================
    for epoch in range(1, EPOCHS + 1):
        # Training Phase
        model.train()
        epoch_loss = 0
        
        for batch_idx, (data, targets) in enumerate(train_loader):
            data = data.to(device)
            targets = targets.to(device)
            
            # Forward pass
            spk_out, mem_out = model(data)
            
            # Average membrane potential over time
            mem_avg = mem_out.mean(dim=0)
            
            # Compute loss
            loss = criterion(mem_avg, targets)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        fold_history['train_loss'].append(avg_loss)
        
        # Validation Phase
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for data, targets in val_loader:
                data = data.to(device)
                targets = targets.to(device)
                
                # Forward pass
                spk_out, mem_out = model(data)
                
                # Prediction: sum spikes over time
                spk_sum = spk_out.sum(dim=0)
                _, predicted = spk_sum.max(1)
                
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        
        val_acc = correct / total
        fold_history['val_acc'].append(val_acc)
        
        # Save best model for this fold
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), snn_dir / f'snn_fold_{fold}.pth')
        
        # Print progress (less frequent to avoid clutter)
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | Best: {best_acc:.4f}")
    
    # ========================
    # Final Evaluation for this Fold
    # ========================
    model.load_state_dict(torch.load(snn_dir / f'snn_fold_{fold}.pth'))
    model.eval()
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for data, targets in val_loader:
            data = data.to(device)
            targets = targets.to(device)
            
            spk_out, mem_out = model(data)
            spk_sum = spk_out.sum(dim=0)
            _, predicted = spk_sum.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    # Calculate Fall_Initiation recall
    fall_init_idx = label_map["Fall_Initiation"]
    fall_init_mask = all_targets == fall_init_idx
    fall_init_recall = (all_preds[fall_init_mask] == fall_init_idx).mean()
    
    # Store results
    fold_results['val_acc'].append(best_acc)
    fold_results['fall_init_recall'].append(fall_init_recall)
    fold_results['train_history'].append(fold_history)
    fold_results['predictions'].append(all_preds)
    fold_results['targets'].append(all_targets)
    
    print(f"\n  Fold {fold} Results:")
    print(f"    Best Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
    print(f"    Fall_Init Recall: {fall_init_recall:.4f} ({fall_init_recall*100:.2f}%)")

# ========================
# Aggregate Results Across All Folds
# ========================
print("\n" + "="*80)
print("5-FOLD CROSS-VALIDATION RESULTS")
print("="*80)

mean_acc = np.mean(fold_results['val_acc'])
std_acc = np.std(fold_results['val_acc'])
mean_recall = np.mean(fold_results['fall_init_recall'])
std_recall = np.std(fold_results['fall_init_recall'])

print(f"\nAccuracy per fold:")
for i, acc in enumerate(fold_results['val_acc'], 1):
    print(f"  Fold {i}: {acc:.4f} ({acc*100:.2f}%)")

print(f"\nMean Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean Fall_Init Recall: {mean_recall:.4f} ± {std_recall:.4f} ({mean_recall*100:.2f}% ± {std_recall*100:.2f}%)")

# ========================
# CNN vs SNN Comparison
# ========================
print("\n" + "="*80)
print("CNN vs SNN COMPARISON (5-FOLD AVERAGE)")
print("="*80)
print(f"{'Metric':<25} {'CNN (FP32)':<20} {'SNN (Trained)':<20} {'Drop':<10}")
print("-"*75)
print(f"{'Accuracy':<25} {94.71:<20.2f}% {mean_acc*100:.2f}% ± {std_acc*100:.2f}% {94.71 - mean_acc*100:<10.2f}%")
print(f"{'Fall_Init Recall':<25} {97.82:<20.2f}% {mean_recall*100:.2f}% ± {std_recall*100:.2f}% {97.82 - mean_recall*100:<10.2f}%")
print("-"*75)

# Performance assessment
accuracy_drop = 94.71 - mean_acc*100
if accuracy_drop < 7:
    print("\n✅ EXCELLENT! Average accuracy drop < 7%")
elif accuracy_drop < 10:
    print("\n✅ GOOD! Average accuracy drop < 10%")
elif accuracy_drop < 15:
    print("\n⚠️  ACCEPTABLE! Average accuracy drop < 15%")
else:
    print("\n❌ NEEDS TUNING! Average accuracy drop > 15%")

# ========================
# Aggregate Classification Report
# ========================
print("\n" + "="*80)
print("AGGREGATE CLASSIFICATION REPORT (ALL FOLDS)")
print("="*80)

# Concatenate all predictions and targets
all_fold_preds = np.concatenate(fold_results['predictions'])
all_fold_targets = np.concatenate(fold_results['targets'])

class_names = [reverse_label_map[i] for i in range(6)]
report = classification_report(all_fold_targets, all_fold_preds, target_names=class_names, digits=4)
print(report)

# ========================
# Save Summary
# ========================
summary = {
    'mean_accuracy': mean_acc,
    'std_accuracy': std_acc,
    'mean_fall_init_recall': mean_recall,
    'std_fall_init_recall': std_recall,
    'fold_accuracies': fold_results['val_acc'],
    'fold_recalls': fold_results['fall_init_recall']
}

import json
with open(snn_dir / 'cv_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Models saved: {snn_dir / 'snn_fold_[1-5].pth'}")
print(f"✅ Summary saved: {snn_dir / 'cv_results.json'}")

# %%

In [ ]:
# %% [markdown]
# ## FallNet Phase 2: LMU-SNN Hybrid
#
# Architecture:
#   IMU [batch, 6, 200]
#     → LMUEncoder (one LMUCell per IMU channel, processes 200 timesteps)
#     → [batch, lmu_hidden × 6]  compact Legendre temporal encoding
#     → Spiking FC classifier (3 × Linear → BN → LIF, integrated over num_steps)
#     → Spike accumulation → [batch, 6] class logits
#
# Theory: Fourier → Wavelets → LMU (Padé/Legendre basis) → SNN (LIF integration)

# %%
# --- Cell 1: Imports ---

import torch
import torch.nn as nn
import torch.nn.functional as F
import snntorch as snn
from snntorch import surrogate
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import numpy as np
from scipy.signal import cont2discrete

print("✅ Imports OK")

# %%
# --- Cell 2: LMU Cell ---

class LMUCell(nn.Module):
    """
    Single-step Legendre Memory Unit.

    Per timestep:
        e_t  = tanh(W_x @ x_t + W_h @ h_{t-1} + W_m @ m_{t-1})   # encoder
        m_t  = A_bar @ m_{t-1} + B_bar @ e_t                       # memory update
        h_t  = tanh(H_x @ x_t + H_m @ m_t)                        # hidden state

    A_bar, B_bar are analytically derived (Voelker et al. 2019) via
    zero-order-hold discretisation of the continuous Legendre delay ODE.
    They are frozen by default — the network learns only the coupling weights.
    """

    def __init__(self, input_size, hidden_size, order, theta, learn_ab=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.order = order

        # --- Derive A_bar, B_bar analytically ---
        Q = np.arange(order, dtype=np.float64)
        R = (2 * Q + 1)[:, None]
        j, i = np.meshgrid(Q, Q)
        A_cont = np.where(i < j, -1, (-1.0) ** (i - j + 1)) * R / theta
        B_cont = (-1.0) ** Q[:, None] * R / theta

        C = np.zeros((1, order))
        D = np.zeros((1,))
        A_bar, B_bar, _, _, _ = cont2discrete(
            (A_cont, B_cont, C, D), dt=1.0, method='zoh'
        )

        if learn_ab:
            self.A_bar = nn.Parameter(torch.FloatTensor(A_bar))
            self.B_bar = nn.Parameter(torch.FloatTensor(B_bar))
        else:
            self.register_buffer('A_bar', torch.FloatTensor(A_bar))
            self.register_buffer('B_bar', torch.FloatTensor(B_bar))

        # Encoder weights
        self.e_x = nn.Linear(input_size,  1, bias=False)
        self.e_h = nn.Linear(hidden_size, 1, bias=False)
        self.e_m = nn.Linear(order,       1, bias=False)

        # Hidden weights
        self.h_x = nn.Linear(input_size, hidden_size, bias=False)
        self.h_m = nn.Linear(order,      hidden_size, bias=False)

    def forward(self, x, state):
        h, m = state
        u     = torch.tanh(self.e_x(x) + self.e_h(h) + self.e_m(m))   # [B, 1]
        m_new = F.linear(m, self.A_bar) + F.linear(u, self.B_bar.T)    # [B, order]
        h_new = torch.tanh(self.h_x(x) + self.h_m(m_new))              # [B, hidden]
        return h_new, (h_new, m_new)

    def init_state(self, batch_size, device):
        return (
            torch.zeros(batch_size, self.hidden_size, device=device),
            torch.zeros(batch_size, self.order,       device=device),
        )


print("✅ LMUCell defined")

# %%
# --- Cell 3: LMU Encoder ---

class LMUEncoder(nn.Module):
    """
    Runs one LMUCell per IMU channel over the full 200-timestep window.
    Each channel independently learns its Legendre memory representation.
    Outputs are concatenated: [batch, hidden_size * in_channels].
    """

    def __init__(self, in_channels=6, hidden_size=32, order=8,
                 theta=200, learn_ab=False):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_size = hidden_size
        self.output_size = hidden_size * in_channels

        self.lmu_cells = nn.ModuleList([
            LMUCell(1, hidden_size, order, theta, learn_ab)
            for _ in range(in_channels)
        ])

    def forward(self, x):
        # x: [batch, in_channels, seq_len]
        batch_size = x.size(0)
        device     = x.device

        states = [cell.init_state(batch_size, device) for cell in self.lmu_cells]

        for t in range(x.size(2)):                          # step over 200 timesteps
            for c, cell in enumerate(self.lmu_cells):
                _, states[c] = cell(x[:, c:c+1, t], states[c])

        h_finals = [states[c][0] for c in range(self.in_channels)]
        return torch.cat(h_finals, dim=-1)                  # [batch, hidden * channels]


print("✅ LMUEncoder defined")

# %%
# --- Cell 4: FallNet LMU-SNN Model ---

class FallNet_LMU_SNN(nn.Module):
    """
    Stage 1 (LMU):  Encodes 200-timestep IMU window into Legendre state
    Stage 2 (SNN):  Integrates that state over num_steps via spiking LIF layers
    """

    def __init__(
        self,
        num_classes = 6,
        num_steps   = 25,
        lmu_hidden  = 32,    # hidden size per channel → 192 total (32×6)
        lmu_order   = 8,     # Legendre polynomial order
        lmu_theta   = 200,   # match your sequence length
        beta        = 0.95,
        threshold   = 1.0,
        learn_ab    = False,
    ):
        super().__init__()
        self.num_steps = num_steps
        spike_grad = surrogate.fast_sigmoid(slope=25)

        # Stage 1: LMU encoder
        self.lmu_encoder = LMUEncoder(
            in_channels=6,
            hidden_size=lmu_hidden,
            order=lmu_order,
            theta=lmu_theta,
            learn_ab=learn_ab,
        )
        lmu_out = lmu_hidden * 6  # 192 with defaults

        # Stage 2: Spiking classifier
        self.fc1  = nn.Linear(lmu_out, 128)
        self.bn1  = nn.BatchNorm1d(128)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad,
                               threshold=threshold, learn_beta=True)

        self.fc2  = nn.Linear(128, 64)
        self.bn2  = nn.BatchNorm1d(64)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad,
                               threshold=threshold, learn_beta=True)

        self.fc3  = nn.Linear(64, num_classes)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad,
                               threshold=threshold, learn_beta=True)

    def forward(self, x):
        # LMU encoding runs once (not per SNN step)
        lmu_out = self.lmu_encoder(x)           # [batch, 192]

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()

        spk_rec, mem_rec = [], []

        for _ in range(self.num_steps):
            spk1, mem1 = self.lif1(self.bn1(self.fc1(lmu_out)), mem1)
            spk2, mem2 = self.lif2(self.bn2(self.fc2(spk1)),    mem2)
            spk3, mem3 = self.lif3(self.fc3(spk2),              mem3)
            spk_rec.append(spk3)
            mem_rec.append(mem3)

        return torch.stack(spk_rec), torch.stack(mem_rec)


# Sanity check
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_model     = FallNet_LMU_SNN().to(device)
_x         = torch.randn(4, 6, 200).to(device)
with torch.no_grad():
    _spk, _mem = _model(_x)

total_p = sum(p.numel() for p in _model.parameters())
lmu_p   = sum(p.numel() for p in _model.lmu_encoder.parameters())
print(f"✅ FallNet_LMU_SNN defined")
print(f"   Input:  {list(_x.shape)}")
print(f"   Output: {list(_spk.shape)}  [steps, batch, classes]")
print(f"   Params: {total_p:,} total  ({lmu_p:,} LMU + {total_p-lmu_p:,} SNN)")
del _model, _x, _spk, _mem

# %%
# --- Cell 5: Dataset + Class Weights ---

class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Assumes X_data_torch, y_labels_torch already defined from your preprocessing cell
cw = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(6),
    y=y_labels_torch.numpy()
)
cw = np.clip(cw, None, 3.0)
cw_tensor = torch.FloatTensor(cw).to(device)

print("Class weights (capped at 3×):")
for i, w in enumerate(cw):
    print(f"  {reverse_label_map[i]:<30s}: {w:.3f}×")

dataset = FallDataset(X_data_torch, y_labels_torch)
print(f"\n✅ Dataset ready: {len(dataset):,} samples")

# %%
# --- Cell 6: Training Config ---

BATCH_SIZE = 32     # smaller than pure-SNN due to LMU sequential cost
EPOCHS     = 30
LR         = 5e-4
NUM_STEPS  = 25
LMU_HIDDEN = 32     # 32 × 6 channels = 192 features into SNN
LMU_ORDER  = 8      # Legendre polynomial order; try 4 or 16 to tune
N_FOLDS    = 5

lmu_snn_dir = models_dir / 'lmu_snn'
lmu_snn_dir.mkdir(exist_ok=True)

print("=" * 80)
print("FallNet LMU-SNN — 5-Fold Cross-Validation")
print("=" * 80)
print(f"Device:     {device}")
print(f"Batch:      {BATCH_SIZE} | Epochs: {EPOCHS} | SNN steps: {NUM_STEPS}")
print(f"LMU hidden: {LMU_HIDDEN}/channel | LMU order: {LMU_ORDER}")
print(f"LMU output: {LMU_HIDDEN * 6} features → SNN")

# %%
# --- Cell 7: 5-Fold Training Loop ---

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = {
    'val_acc':          [],
    'fall_init_recall': [],
    'predictions':      [],
    'targets':          [],
}

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_data_torch, y_labels_torch), 1
):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{N_FOLDS} — train: {len(train_idx):,}  val: {len(val_idx):,}")
    print(f"{'='*80}")

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True,
    )

    model     = FallNet_LMU_SNN(
        num_steps=NUM_STEPS, lmu_hidden=LMU_HIDDEN, lmu_order=LMU_ORDER
    ).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        # ---- Train ----
        model.train()
        epoch_loss = 0.0

        for data, targets in train_loader:
            data, targets = data.to(device), targets.to(device)
            spk_out, mem_out = model(data)

            # Combined loss: membrane average + 0.5 × spike count
            loss = (criterion(mem_out.mean(dim=0), targets)
                    + 0.5 * criterion(spk_out.sum(dim=0), targets))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()

        # ---- Validate ----
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for data, targets in val_loader:
                data, targets = data.to(device), targets.to(device)
                predicted = model(data)[0].sum(dim=0).argmax(dim=1)
                total    += targets.size(0)
                correct  += (predicted == targets).sum().item()

        val_acc = correct / total
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(),
                       lmu_snn_dir / f'lmu_snn_fold_{fold}.pth')

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:2d}/{EPOCHS} | "
                  f"Loss: {epoch_loss/len(train_loader):.4f} | "
                  f"Val: {val_acc:.4f} | Best: {best_acc:.4f}")

    # ---- Final fold eval ----
    model.load_state_dict(torch.load(
        lmu_snn_dir / f'lmu_snn_fold_{fold}.pth', weights_only=True
    ))
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for data, targets in val_loader:
            predicted = model(data.to(device))[0].sum(dim=0).argmax(dim=1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.numpy())

    all_preds   = np.array(all_preds)
    all_targets = np.array(all_targets)

    fi_idx    = label_map['Fall_Initiation']
    fi_mask   = all_targets == fi_idx
    fi_recall = (all_preds[fi_mask] == fi_idx).mean() if fi_mask.any() else 0.0

    fold_results['val_acc'].append(best_acc)
    fold_results['fall_init_recall'].append(fi_recall)
    fold_results['predictions'].append(all_preds)
    fold_results['targets'].append(all_targets)

    print(f"\n  Fold {fold} → Acc: {best_acc*100:.2f}%  "
          f"Fall_Init Recall: {fi_recall*100:.2f}%")

# %%
# --- Cell 8: Results ---

mean_acc    = np.mean(fold_results['val_acc'])
std_acc     = np.std(fold_results['val_acc'])
mean_recall = np.mean(fold_results['fall_init_recall'])
std_recall  = np.std(fold_results['fall_init_recall'])

print("=" * 80)
print("5-FOLD RESULTS — FallNet LMU-SNN")
print("=" * 80)
for i, (a, r) in enumerate(zip(
    fold_results['val_acc'], fold_results['fall_init_recall']
), 1):
    print(f"  Fold {i}: Acc {a*100:.2f}%  Fall_Init Recall {r*100:.2f}%")

print(f"\nMean Accuracy:        {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Mean Fall_Init Recall:{mean_recall*100:.2f}% ± {std_recall*100:.2f}%")

print("\n" + "=" * 80)
print("COMPARISON")
print("=" * 80)
print(f"{'Model':<25} {'Accuracy':<20} {'Fall_Init Recall'}")
print("-" * 65)
print(f"{'CNN (FP32)':<25} {'94.71%':<20} {'97.82%'}")
print(f"{'SNN (trained)':<25} {'88.83%':<20} {'see cv_results.json'}")
print(f"{'LMU-SNN':<25} "
      f"{mean_acc*100:.2f}% ± {std_acc*100:.2f}%    "
      f"{mean_recall*100:.2f}% ± {std_recall*100:.2f}%")

print("\n" + "=" * 80)
print("AGGREGATE CLASSIFICATION REPORT")
print("=" * 80)
all_p = np.concatenate(fold_results['predictions'])
all_t = np.concatenate(fold_results['targets'])
names = [reverse_label_map[i] for i in range(6)]
print(classification_report(all_t, all_p, target_names=names, digits=4))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score# At the very start of comparison script, BEFORE the fold loop:
print("\n" + "="*80)
print("TESTING SNN MODEL DIRECTLY")
print("="*80)

# Load ONE model
test_model = MicroCNN_SNN(6, 25).to(device)
test_model.load_state_dict(torch.load(snn_dir / "snn_fold_1.pth"))
test_model.eval()

# Get fold 1 validation data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
_, val_idx = next(skf.split(X_data, y_labels))

X_val = X_data[val_idx]
y_val = y_labels[val_idx]

# Method 1: Manual batch loop (what your comparison script does)
X_val_torch = torch.FloatTensor(X_val).permute(0, 2, 1).to(device)
all_preds_manual = []

batch_size = 32
with torch.no_grad():
    for i in range(0, len(X_val_torch), batch_size):
        batch = X_val_torch[i:i+batch_size]
        spk_out, mem_out = test_model(batch)
        _, predicted = spk_out.sum(dim=0).max(1)
        all_preds_manual.extend(predicted.cpu().numpy())

acc_manual = accuracy_score(y_val, np.array(all_preds_manual))

# Method 2: All at once (what worked in debug)
with torch.no_grad():
    spk_out, mem_out = test_model(X_val_torch)
    _, predicted_all = spk_out.sum(dim=0).max(1)

acc_all = accuracy_score(y_val, predicted_all.cpu().numpy())

print(f"Method 1 (batch loop): {acc_manual:.4f}")
print(f"Method 2 (all at once): {acc_all:.4f}")
print(f"Expected: ~0.88")

if abs(acc_manual - acc_all) > 0.01:
    print("\n❌ METHODS DIFFER! Bug is in batch loop!")
else:
    print("\n✅ Methods match! Bug is elsewhere!")

print("="*80)


In [ ]:
# Setup paths
current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:
        project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
        break

data_dir = project_root / "fall_detection_data"
processed_dir = data_dir / "processed"
models_dir = data_dir / "models"
micro_cnn_dir = models_dir / "micro_cnn"
snn_dir = models_dir / "snn"

print(f"\n📂 Directories:")
print(f"   Micro-CNN: {micro_cnn_dir}")
print(f"   SNN:       {snn_dir}")

# Check what files exist
if micro_cnn_dir.exists():
    cnn_files = list(micro_cnn_dir.glob("*.keras"))
    print(f"\n   Found {len(cnn_files)} CNN models:")
    for f in sorted(cnn_files):
        print(f"     - {f.name}")
else:
    print(f"   ⚠️  {micro_cnn_dir} doesn't exist")

if snn_dir.exists():
    snn_files = list(snn_dir.glob("*.pth"))
    print(f"\n   Found {len(snn_files)} SNN models:")
    for f in sorted(snn_files):
        print(f"     - {f.name}")
else:
    print(f"   ⚠️  {snn_dir} doesn't exist")

# Load data
X_data = np.load(processed_dir / "X_data_6class.npy")
y_labels = np.load(processed_dir / "y_labels_6class.npy")

with open(processed_dir / "label_map_6class.json", 'r') as f:
    label_map = json.load(f)

reverse_label_map = {v: k for k, v in label_map.items()}

print(f"\n📊 Data:")
print(f"   X shape: {X_data.shape}")
print(f"   y shape: {y_labels.shape}")

# %% [markdown]
## SNN Architecture (for loading)

# %%
class MicroCNN_SNN(nn.Module):
    """Clean SNN implementation without manual detaching"""
    
    def __init__(self, num_classes=6, num_steps=25, beta=0.95, threshold=1.0):
        super().__init__()
        
        self.num_classes = num_classes
        self.num_steps = num_steps
        
        # Surrogate gradient for backprop
        spike_grad = surrogate.fast_sigmoid(slope=25)
        
        # Block 1: Conv -> BN -> LIF -> Pool
        self.conv1 = nn.Conv1d(6, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool1 = nn.MaxPool1d(2)
        
        # Block 2: Conv -> BN -> LIF -> Pool
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool2 = nn.MaxPool1d(2)
        
        # Block 3: Conv -> BN -> LIF -> Pool
        self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool3 = nn.MaxPool1d(2)
        
        # Global pooling
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128, 64)
        self.lif4 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        
        self.fc2 = nn.Linear(64, num_classes)
        self.lif_out = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
    
    def forward(self, x):
        batch_size = x.size(0)
    
        # Initialize membrane potentials
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()
        mem_out = self.lif_out.init_leaky()
    
        spk_rec = []
        mem_rec = []
    
        for _ in range(self.num_steps):
            # Block 1
            cur1 = self.pool1(self.bn1(self.conv1(x)))
            spk1, mem1 = self.lif1(cur1, mem1)
        
            # Block 2
            cur2 = self.pool2(self.bn2(self.conv2(spk1)))
            spk2, mem2 = self.lif2(cur2, mem2)
        
            # Block 3
            cur3 = self.pool3(self.bn3(self.conv3(spk2)))
            spk3, mem3 = self.lif3(cur3, mem3)
        
            # Global pool + flatten
            cur4 = self.global_pool(spk3).squeeze(-1)
            cur4 = self.fc1(cur4)
            spk4, mem4 = self.lif4(cur4, mem4)
        
            # Output layer
            cur_out = self.fc2(spk4)
            spk_out, mem_out = self.lif_out(cur_out, mem_out)
        
            spk_rec.append(spk_out)
            mem_rec.append(mem_out)
    
            return torch.stack(spk_rec), torch.stack(mem_rec)

print("✅ SNN architecture defined")

# %% [markdown]
## Evaluate All Models

# %%
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

# K-Fold split (same as training)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

micro_cnn_results = []
snn_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"EVALUATING FOLD {fold}/5")
    print(f"{'='*80}")
    
    # Get validation data
    X_val = X_data[val_idx]
    y_val = y_labels[val_idx]
    
    print(f"Validation samples: {len(X_val):,}")
    
    # ================================================================
    # Evaluate Micro-CNN
    # ================================================================
    micro_cnn_path = micro_cnn_dir / f"micro_cnn_fold_{fold}.keras"
    
    if micro_cnn_path.exists():
        print(f"\n--- Micro-CNN Fold {fold} ---")
        
        # Load Keras model
        model_cnn = keras.models.load_model(micro_cnn_path)
        
        # Predict
        y_pred_cnn = np.argmax(model_cnn.predict(X_val, verbose=0), axis=1)
        
        # Metrics
        acc_cnn = accuracy_score(y_val, y_pred_cnn)
        prec_cnn = precision_score(y_val, y_pred_cnn, average='weighted', zero_division=0)
        rec_cnn = recall_score(y_val, y_pred_cnn, average='weighted', zero_division=0)
        f1_cnn = f1_score(y_val, y_pred_cnn, average='weighted', zero_division=0)
        
        # Fall_Initiation recall
        fall_init_idx = label_map["Fall_Initiation"]
        fall_init_mask = y_val == fall_init_idx
        if fall_init_mask.sum() > 0:
            fall_init_recall_cnn = (y_pred_cnn[fall_init_mask] == fall_init_idx).mean()
        else:
            fall_init_recall_cnn = 0.0
        
        print(f"  Accuracy:  {acc_cnn:.4f} ({acc_cnn*100:.2f}%)")
        print(f"  Precision: {prec_cnn:.4f}")
        print(f"  Recall:    {rec_cnn:.4f}")
        print(f"  F1-Score:  {f1_cnn:.4f}")
        print(f"  Fall_Init Recall: {fall_init_recall_cnn:.4f} ({fall_init_recall_cnn*100:.2f}%)")
        
        micro_cnn_results.append({
            'fold': fold,
            'accuracy': acc_cnn,
            'precision': prec_cnn,
            'recall': rec_cnn,
            'f1': f1_cnn,
            'fall_init_recall': fall_init_recall_cnn,
            'y_true': y_val,
            'y_pred': y_pred_cnn
        })
    else:
        print(f"\n⚠️  Micro-CNN Fold {fold}: Not found at {micro_cnn_path}")
    
    # ================================================================
    # Evaluate SNN
    # ================================================================
    # Try both possible filenames
    # ✅ FIXED:
    snn_path = snn_dir / f"snn_fold_{fold}.pth"

    
    if snn_path.exists():
        print(f"\n--- SNN Fold {fold} ---")
        print(f"  Loading from: {snn_path.name}")
        
        # Load PyTorch model
        model_snn = MicroCNN_SNN(num_classes=6, num_steps=25).to(device)
        model_snn.load_state_dict(torch.load(snn_path, map_location=device))
        model_snn.eval()
        
        # Convert data to PyTorch
        X_val_torch = torch.FloatTensor(X_val).permute(0, 2, 1).to(device)
        
        # Predict
        all_preds_snn = []
        batch_size = 32
        
        with torch.no_grad():
            for i in range(0, len(X_val_torch), batch_size):
                batch = X_val_torch[i:i+batch_size]
                spk_out, mem_out = model_snn(batch)
                
                # Sum spikes over time
                _, predicted = spk_out.sum(dim=0).max(1)
                all_preds_snn.extend(predicted.cpu().numpy())
        
        y_pred_snn = np.array(all_preds_snn)
        
        # Metrics
        acc_snn = accuracy_score(y_val, y_pred_snn)
        prec_snn = precision_score(y_val, y_pred_snn, average='weighted', zero_division=0)
        rec_snn = recall_score(y_val, y_pred_snn, average='weighted', zero_division=0)
        f1_snn = f1_score(y_val, y_pred_snn, average='weighted', zero_division=0)
        
        # Fall_Initiation recall
        if fall_init_mask.sum() > 0:
            fall_init_recall_snn = (y_pred_snn[fall_init_mask] == fall_init_idx).mean()
        else:
            fall_init_recall_snn = 0.0
        
        print(f"  Accuracy:  {acc_snn:.4f} ({acc_snn*100:.2f}%)")
        print(f"  Precision: {prec_snn:.4f}")
        print(f"  Recall:    {rec_snn:.4f}")
        print(f"  F1-Score:  {f1_snn:.4f}")
        print(f"  Fall_Init Recall: {fall_init_recall_snn:.4f} ({fall_init_recall_snn*100:.2f}%)")
        
        snn_results.append({
            'fold': fold,
            'accuracy': acc_snn,
            'precision': prec_snn,
            'recall': rec_snn,
            'f1': f1_snn,
            'fall_init_recall': fall_init_recall_snn,
            'y_true': y_val,
            'y_pred': y_pred_snn
        })
    else:
        print(f"\n⚠️  SNN Fold {fold}: Not found")

# %% [markdown]
## Summary Statistics

# %%
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

# Micro-CNN Summary
if micro_cnn_results:
    print("\n--- Micro-CNN Results ---")
    
    cnn_accs = [r['accuracy'] for r in micro_cnn_results]
    cnn_f1s = [r['f1'] for r in micro_cnn_results]
    cnn_fall_recalls = [r['fall_init_recall'] for r in micro_cnn_results]
    
    print(f"Folds evaluated: {len(micro_cnn_results)}/5")
    print(f"Accuracy:        {np.mean(cnn_accs):.4f} ± {np.std(cnn_accs):.4f} ({np.mean(cnn_accs)*100:.2f}%)")
    print(f"F1-Score:        {np.mean(cnn_f1s):.4f} ± {np.std(cnn_f1s):.4f}")
    print(f"Fall_Init Recall: {np.mean(cnn_fall_recalls):.4f} ± {np.std(cnn_fall_recalls):.4f} ({np.mean(cnn_fall_recalls)*100:.2f}%)")
else:
    print("\n❌ No Micro-CNN results found!")

# SNN Summary
if snn_results:
    print("\n--- SNN Results ---")
    
    snn_accs = [r['accuracy'] for r in snn_results]
    snn_f1s = [r['f1'] for r in snn_results]
    snn_fall_recalls = [r['fall_init_recall'] for r in snn_results]
    
    print(f"Folds evaluated: {len(snn_results)}/5")
    print(f"Accuracy:        {np.mean(snn_accs):.4f} ± {np.std(snn_accs):.4f} ({np.mean(snn_accs)*100:.2f}%)")
    print(f"F1-Score:        {np.mean(snn_f1s):.4f} ± {np.std(snn_f1s):.4f}")
    print(f"Fall_Init Recall: {np.mean(snn_fall_recalls):.4f} ± {np.std(snn_fall_recalls):.4f} ({np.mean(snn_fall_recalls)*100:.2f}%)")
else:
    print("\n❌ No SNN results found!")

# %% [markdown]
## Comparison Table

# %%
if micro_cnn_results and snn_results:
    print("\n" + "="*80)
    print("MICRO-CNN vs SNN COMPARISON")
    print("="*80)
    
    # Per-fold comparison
    comparison_data = []
    
    for fold in range(1, 6):
        cnn_fold = next((r for r in micro_cnn_results if r['fold'] == fold), None)
        snn_fold = next((r for r in snn_results if r['fold'] == fold), None)
        
        if cnn_fold and snn_fold:
            comparison_data.append({
                'Fold': fold,
                'CNN_Acc': f"{cnn_fold['accuracy']:.4f}",
                'SNN_Acc': f"{snn_fold['accuracy']:.4f}",
                'Diff': f"{(cnn_fold['accuracy'] - snn_fold['accuracy']):.4f}",
                'CNN_Fall': f"{cnn_fold['fall_init_recall']:.4f}",
                'SNN_Fall': f"{snn_fold['fall_init_recall']:.4f}"
            })
    
    if comparison_data:
        df = pd.DataFrame(comparison_data)
        print("\nPer-Fold Results:")
        print(df.to_string(index=False))
        
        # Overall comparison
        print(f"\n{'='*80}")
        print("OVERALL COMPARISON")
        print(f"{'='*80}")
        print(f"                        Micro-CNN       SNN             Difference")
        print(f"{'-'*80}")
        print(f"Accuracy:               {np.mean(cnn_accs):.4f}          {np.mean(snn_accs):.4f}          {np.mean(cnn_accs) - np.mean(snn_accs):.4f}")
        print(f"Fall_Init Recall:       {np.mean(cnn_fall_recalls):.4f}          {np.mean(snn_fall_recalls):.4f}          {np.mean(cnn_fall_recalls) - np.mean(snn_fall_recalls):.4f}")
        print(f"{'-'*80}")
        
        drop_pct = ((np.mean(cnn_accs) - np.mean(snn_accs)) / np.mean(cnn_accs)) * 100
        
        if drop_pct > 0:
            print(f"\nCNN is {drop_pct:.2f}% better than SNN")
        else:
            print(f"\nSNN is {-drop_pct:.2f}% better than CNN")

# %% [markdown]
## Visualization

# %%
if micro_cnn_results and snn_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Get matching folds
    folds = [r['fold'] for r in micro_cnn_results if any(s['fold'] == r['fold'] for s in snn_results)]
    cnn_vals = [r['accuracy']*100 for r in micro_cnn_results if r['fold'] in folds]
    snn_vals = [r['accuracy']*100 for r in snn_results if r['fold'] in folds]
    
    x = np.arange(len(folds))
    width = 0.35
    
    # Plot 1: Accuracy
    axes[0].bar(x - width/2, cnn_vals, width, label='Micro-CNN', alpha=0.8, color='#2ecc71')
    axes[0].bar(x + width/2, snn_vals, width, label='SNN', alpha=0.8, color='#e74c3c')
    
    axes[0].set_xlabel('Fold', fontsize=12)
    axes[0].set_ylabel('Accuracy (%)', fontsize=12)
    axes[0].set_title('Accuracy by Fold', fontsize=13, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(folds)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Fall_Init recall
    cnn_fall_vals = [r['fall_init_recall']*100 for r in micro_cnn_results if r['fold'] in folds]
    snn_fall_vals = [r['fall_init_recall']*100 for r in snn_results if r['fold'] in folds]
    
    axes[1].bar(x - width/2, cnn_fall_vals, width, label='Micro-CNN', alpha=0.8, color='#2ecc71')
    axes[1].bar(x + width/2, snn_fall_vals, width, label='SNN', alpha=0.8, color='#e74c3c')
    
    axes[1].set_xlabel('Fold', fontsize=12)
    axes[1].set_ylabel('Fall_Initiation Recall (%)', fontsize=12)
    axes[1].set_title('Fall Detection Recall by Fold', fontsize=13, fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(folds)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(models_dir / 'cnn_vs_snn_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Plot saved: {models_dir / 'cnn_vs_snn_comparison.png'}")

# Save summary
if micro_cnn_results or snn_results:
    summary_lines = ["MODEL COMPARISON SUMMARY", "="*80, ""]
    
    if micro_cnn_results:
        summary_lines.extend([
            "Micro-CNN Results:",
            f"  Folds: {len(micro_cnn_results)}/5",
            f"  Accuracy:  {np.mean(cnn_accs):.4f} ± {np.std(cnn_accs):.4f} ({np.mean(cnn_accs)*100:.2f}%)",
            f"  Fall_Init: {np.mean(cnn_fall_recalls):.4f} ± {np.std(cnn_fall_recalls):.4f} ({np.mean(cnn_fall_recalls)*100:.2f}%)",
            ""
        ])
    
    if snn_results:
        summary_lines.extend([
            "SNN Results:",
            f"  Folds: {len(snn_results)}/5",
            f"  Accuracy:  {np.mean(snn_accs):.4f} ± {np.std(snn_accs):.4f} ({np.mean(snn_accs)*100:.2f}%)",
            f"  Fall_Init: {np.mean(snn_fall_recalls):.4f} ± {np.std(snn_fall_recalls):.4f} ({np.mean(snn_fall_recalls)*100:.2f}%)",
            ""
        ])
    
    if micro_cnn_results and snn_results:
        summary_lines.extend([
            "Difference:",
            f"  Accuracy:  {np.mean(cnn_accs) - np.mean(snn_accs):.4f} ({((np.mean(cnn_accs) - np.mean(snn_accs))/np.mean(cnn_accs))*100:.2f}%)",
            f"  Fall_Init: {np.mean(cnn_fall_recalls) - np.mean(snn_fall_recalls):.4f}"
        ])
    
    summary = "\n".join(summary_lines)
    
    print("\n" + summary)
    
    with open(models_dir / 'model_comparison_summary.txt', 'w') as f:
        f.write(summary)
    
    print(f"\n✅ Summary saved: {models_dir / 'model_comparison_summary.txt'}")

print("\n" + "="*80)
print("COMPARISON COMPLETE")
print("="*80)

# %%

In [ ]:
# %% [markdown]
# # Convert PyTorch SNN to Nengo and Deploy to Arduino
# 
# This notebook:
# 1. Builds your SNN architecture in Nengo
# 2. Trains with real LIF neurons
# 3. Exports to TFLite Float16
# 4. Verifies accuracy
# 5. Prepares for Arduino deployment

# %%
import numpy as np
import nengo
import nengo_dl
import tensorflow as tf
from pathlib import Path
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import warnings
from onnx import shape_inference
warnings.filterwarnings('ignore')

print("="*80)
print("NENGO SNN CONVERSION & DEPLOYMENT")
print("="*80)

# Check Nengo version
import nengo
print(f"\nNengo version: {nengo.__version__}")

try:
    import nengo_dl
    print(f"Nengo-DL version: {nengo_dl.__version__}")
except ImportError:
    print("\n⚠️  Nengo-DL not installed!")
    print("Run: pip install nengo nengo-dl")
    raise

# %%
# Setup paths
current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:
        project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
        break

data_dir = project_root / "fall_detection_data"
processed_dir = data_dir / "processed"
models_dir = data_dir / "models"
nengo_dir = models_dir / "nengo_snn"
nengo_dir.mkdir(exist_ok=True)

print(f"\n📂 Directories:")
print(f"   Data: {processed_dir}")
print(f"   Output: {nengo_dir}")

# %%
# Load data
X_data = np.load(processed_dir / "X_data_6class.npy")
y_labels = np.load(processed_dir / "y_labels_6class.npy")

with open(processed_dir / "label_map_6class.json", 'r') as f:
    label_map = json.load(f)

reverse_label_map = {v: k for k, v in label_map.items()}

print(f"\n📊 Data loaded:")
print(f"   X shape: {X_data.shape}")
print(f"   y shape: {y_labels.shape}")
print(f"   Classes: {list(label_map.keys())}")

# %% [markdown]
## Prepare Data for Nengo

# %%
# Nengo-DL expects specific input format
# We need to flatten the input for now (can extend to temporal processing later)
print("\n" + "="*80)
print("PREPARING DATA FOR NENGO")
print("="*80)

# Flatten [batch, 200, 6] → [batch, 1200]
X_flat = X_data.reshape(len(X_data), -1)
print(f"\nFlattened input shape: {X_flat.shape}")

# Convert labels to one-hot
y_onehot = tf.keras.utils.to_categorical(y_labels, 6)
print(f"One-hot labels shape: {y_onehot.shape}")

# Split into train/val (Fold 1 for now)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(skf.split(X_data, y_labels))

X_train = X_flat[train_idx].astype(np.float32)
y_train = y_onehot[train_idx].astype(np.float32)
X_val = X_flat[val_idx].astype(np.float32)
y_val = y_onehot[val_idx].astype(np.float32)

print(f"\n📊 Fold 1 split:")
print(f"   Train: {len(X_train):,} samples")
print(f"   Val:   {len(X_val):,} samples")

# %% [markdown]
## Build Nengo SNN Architecture
# 
# We'll build an SNN that matches your PyTorch architecture:
# - Conv layers → LIF neurons → Pooling (3 blocks)
# - Global pooling
# - Dense → LIF → Dense

# %%
print("\n" + "="*80)
print("BUILDING NENGO SNN")
print("="*80)

def build_nengo_snn(input_size=1200, num_classes=6):
    """
    Build SNN in Nengo matching PyTorch architecture
    
    Note: Nengo uses different layer types, so we'll create
    a feedforward SNN with LIF activation functions
    """
    
    with nengo.Network() as net:
        # Configure for training
        nengo_dl.configure_settings(stateful=False)
        
        # Input node
        net.inp = nengo.Node(output=[0] * input_size)
        
        # First hidden layer (equivalent to first conv block)
        net.layer1 = nengo_dl.Layer(tf.keras.layers.Dense(128))(net.inp)
        net.lif1 = nengo_dl.Layer(nengo.LIF(amplitude=0.01))(net.layer1)
        
        # Second hidden layer (equivalent to second conv block)
        net.layer2 = nengo_dl.Layer(tf.keras.layers.Dense(256))(net.lif1)
        net.lif2 = nengo_dl.Layer(nengo.LIF(amplitude=0.01))(net.layer2)
        
        # Third hidden layer (equivalent to third conv block)
        net.layer3 = nengo_dl.Layer(tf.keras.layers.Dense(128))(net.lif2)
        net.lif3 = nengo_dl.Layer(nengo.LIF(amplitude=0.01))(net.layer3)
        
        # Dense layers (equivalent to FC layers)
        net.layer4 = nengo_dl.Layer(tf.keras.layers.Dense(64))(net.lif3)
        net.lif4 = nengo_dl.Layer(nengo.LIF(amplitude=0.01))(net.layer4)
        
        # Output layer
        net.output_layer = nengo_dl.Layer(tf.keras.layers.Dense(num_classes))(net.lif4)
        
        # Probe to read output
        net.output_probe = nengo.Probe(net.output_layer)
    
    return net

# Build the network
net = build_nengo_snn()

print("✅ Nengo SNN architecture built")
print("\nArchitecture:")
print("  Input (1200) → Dense(128) → LIF")
print("              → Dense(256) → LIF")
print("              → Dense(128) → LIF")
print("              → Dense(64)  → LIF")
print("              → Dense(6)   → Output")

# %% [markdown]
## Train Nengo SNN

# %%
print("\n" + "="*80)
print("TRAINING NENGO SNN")
print("="*80)

# Training configuration
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 5e-4

print(f"\nTraining config:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")

# Create simulator
with nengo_dl.Simulator(net, minibatch_size=BATCH_SIZE) as sim:
    print("\n✅ Simulator created")
    
    # Compile
    print("Compiling model...")
    sim.compile(
        optimizer=tf.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.losses.CategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    print("✅ Model compiled")
    
    # Train
    print(f"\nTraining for {EPOCHS} epochs...")
    print("-" * 80)
    
    history = sim.fit(
        {net.inp: X_train},
        {net.output_probe: y_train},
        validation_data=(
            {net.inp: X_val},
            {net.output_probe: y_val}
        ),
        epochs=EPOCHS,
        verbose=2  # Less verbose output
    )
    
    print("-" * 80)
    print("✅ Training complete")
    
    # Final evaluation
    print("\nEvaluating on validation set...")
    test_loss, test_accuracy = sim.evaluate(
        {net.inp: X_val},
        {net.output_probe: y_val},
        verbose=0
    )
    
    print(f"\n{'='*80}")
    print("TRAINING RESULTS")
    print(f"{'='*80}")
    print(f"Validation Loss:     {test_loss:.4f}")
    print(f"Validation Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    
    # Save trained parameters
    params_path = nengo_dir / 'nengo_snn_params.pkl'
    sim.save_params(params_path)
    print(f"\n✅ Model parameters saved: {params_path}")

# %%
# Plot training history
print("\nPlotting training history...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(nengo_dir / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Training plot saved: {nengo_dir / 'training_history.png'}")

# %% [markdown]
## Convert to TFLite (Float16)

# %%
print("\n" + "="*80)
print("CONVERTING TO TFLITE (FLOAT16)")
print("="*80)

with nengo_dl.Simulator(net, minibatch_size=1) as sim:
    # Load trained parameters
    sim.load_params(nengo_dir / 'nengo_snn_params.pkl')
    print("✅ Loaded trained parameters")
    
    # Get converter
    converter = sim.keras_model.export_to_tflite()
    
    # Configure for Float16 quantization
    print("\nConfiguring Float16 quantization...")
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    
    # Convert
    print("Converting to TFLite...")
    tflite_model = converter.convert()
    
    # Save
    tflite_path = nengo_dir / 'nengo_snn_float16.tflite'
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    
    file_size = tflite_path.stat().st_size / 1024
    
    print(f"\n✅ TFLite model saved!")
    print(f"   Path: {tflite_path}")
    print(f"   Size: {file_size:.2f} KB")

# %% [markdown]
## Verify TFLite Model Accuracy

# %%
print("\n" + "="*80)
print("VERIFYING TFLITE MODEL")
print("="*80)

# Load TFLite interpreter
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

# Get input/output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"\nTFLite Model Details:")
print(f"  Input shape:  {input_details[0]['shape']}")
print(f"  Input dtype:  {input_details[0]['dtype']}")
print(f"  Output shape: {output_details[0]['shape']}")
print(f"  Output dtype: {output_details[0]['dtype']}")

# Run inference on validation set
print(f"\nRunning inference on {len(X_val):,} validation samples...")

predictions = []
for i in range(len(X_val)):
    # Set input
    input_data = X_val[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    
    # Run inference
    interpreter.invoke()
    
    # Get output
    output_data = interpreter.get_tensor(output_details[0]['index'])
    pred = np.argmax(output_data[0])
    predictions.append(pred)
    
    # Progress
    if (i + 1) % 500 == 0:
        print(f"  Processed {i+1}/{len(X_val)} samples...")

predictions = np.array(predictions)
y_val_labels = y_labels[val_idx]

# Calculate metrics
tflite_accuracy = accuracy_score(y_val_labels, predictions)
tflite_precision = precision_score(y_val_labels, predictions, average='weighted', zero_division=0)
tflite_recall = recall_score(y_val_labels, predictions, average='weighted', zero_division=0)
tflite_f1 = f1_score(y_val_labels, predictions, average='weighted', zero_division=0)

# Fall_Initiation recall
fall_init_idx = label_map["Fall_Initiation"]
fall_mask = y_val_labels == fall_init_idx
fall_recall = (predictions[fall_mask] == fall_init_idx).mean() if fall_mask.sum() > 0 else 0.0

print(f"\n{'='*80}")
print("TFLITE FLOAT16 RESULTS")
print(f"{'='*80}")
print(f"Accuracy:             {tflite_accuracy:.4f} ({tflite_accuracy*100:.2f}%)")
print(f"Precision:            {tflite_precision:.4f}")
print(f"Recall:               {tflite_recall:.4f}")
print(f"F1-Score:             {tflite_f1:.4f}")
print(f"Fall_Init Recall:     {fall_recall:.4f} ({fall_recall*100:.2f}%)")

# %% [markdown]
## Classification Report

# %%
print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)

class_names = [reverse_label_map[i] for i in range(6)]
report = classification_report(y_val_labels, predictions, target_names=class_names, digits=4)
print("\n" + report)

# %% [markdown]
## Comparison with PyTorch SNN

# %%
print("\n" + "="*80)
print("COMPARISON WITH PYTORCH SNN")
print("="*80)

pytorch_snn_accuracy = 88.83  # From your training results
pytorch_snn_fall_recall = 92.66

print(f"\n{'Model':<25} {'Accuracy':<15} {'Fall_Init Recall':<15}")
print("-" * 55)
print(f"{'PyTorch SNN (FP32)':<25} {pytorch_snn_accuracy:<15.2f}% {pytorch_snn_fall_recall:<15.2f}%")
print(f"{'Nengo SNN (Float16)':<25} {tflite_accuracy*100:<15.2f}% {fall_recall*100:<15.2f}%")
print("-" * 55)

accuracy_diff = abs(pytorch_snn_accuracy - tflite_accuracy*100)
print(f"\nAccuracy difference: {accuracy_diff:.2f}%")

if accuracy_diff < 5:
    print("✅ Nengo SNN matches PyTorch SNN well!")
elif accuracy_diff < 10:
    print("⚠️  Some accuracy drop, but reasonable for conversion")
else:
    print("❌ Significant accuracy drop - may need hyperparameter tuning")

# %% [markdown]
## Deployment Summary

# %%
summary = f"""
NENGO SNN DEPLOYMENT SUMMARY
{'='*80}

Model Architecture:
  - Dense(128) → LIF
  - Dense(256) → LIF
  - Dense(128) → LIF
  - Dense(64)  → LIF
  - Dense(6)   → Output
  
Training:
  - Samples: {len(X_train):,} (Fold 1)
  - Epochs: {EPOCHS}
  - Batch size: {BATCH_SIZE}
  - Learning rate: {LEARNING_RATE}

TFLite Float16 Results:
  - Accuracy:           {tflite_accuracy:.4f} ({tflite_accuracy*100:.2f}%)
  - Precision:          {tflite_precision:.4f}
  - Recall:             {tflite_recall:.4f}
  - F1-Score:           {tflite_f1:.4f}
  - Fall_Init Recall:   {fall_recall:.4f} ({fall_recall*100:.2f}%)
  - Model Size:         {file_size:.2f} KB

Arduino Deployment:
  ✅ TFLite file ready: {tflite_path.name}
  ✅ Size fits Arduino: {file_size:.2f} KB < 976 KB available
  ✅ Uses real LIF neurons (true SNN!)
  
Next Steps:
  1. Flash {tflite_path.name} to Arduino Nano 33 BLE Sense
  2. Measure power consumption during inference
  3. Compare with CNN power measurements
  4. Prove SNNs need neuromorphic hardware!

Files Created:
  - {nengo_dir / 'nengo_snn_params.pkl'} (trained parameters)
  - {nengo_dir / 'nengo_snn_float16.tflite'} (deployment model)
  - {nengo_dir / 'training_history.png'} (training plots)
"""

print("\n" + summary)

# Save summary
with open(nengo_dir / 'deployment_summary.txt', 'w') as f:
    f.write(summary)

print(f"✅ Summary saved: {nengo_dir / 'deployment_summary.txt'}")

# %% [markdown]
## Final Checklist

# %%
print("\n" + "="*80)
print("DEPLOYMENT CHECKLIST")
print("="*80)

checklist = [
    ("✅", "Nengo SNN architecture built"),
    ("✅", "Model trained with real LIF neurons"),
    ("✅", "Converted to TFLite Float16"),
    ("✅", f"Model size verified ({file_size:.2f} KB fits Arduino)"),
    ("✅", f"Accuracy verified ({tflite_accuracy*100:.2f}%)"),
    ("✅", "Ready for Arduino deployment"),
]

for status, item in checklist:
    print(f"  {status} {item}")

print(f"\n{'='*80}")
print("NENGO SNN CONVERSION COMPLETE!")
print(f"{'='*80}")
print(f"\n🎯 Next: Deploy {tflite_path.name} to Arduino and measure power!")

# %%

In [ ]:
import tensorflow as tf
import keras
print(tf.__version__)
print(keras.__version__)

In [ ]:
import sys
print(sys.executable)

In [ ]:
# %% [markdown]
# ## Export SNN to ONNX and TFLite for Arduino Nano BLE Sense

# %%
import os
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
import numpy as np
from pathlib import Path
import onnx
import onnxruntime as ort

# ─── Paths ────────────────────────────────────────────────────────────────────
MODEL_PATH = Path("../fall_detection_data/models/snn/snn_fold_5.pth")
OUTPUT_DIR = Path("../fall_detection_data/models/deployment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ONNX_PATH   = OUTPUT_DIR / "snn_fall.onnx"
TFLITE_PATH = OUTPUT_DIR / "snn_fall.tflite"
TFLITE_INT8 = OUTPUT_DIR / "snn_fall_int8.tflite"

# ─── Model definition (must match training) ───────────────────────────────────
class MicroCNN_SNN(nn.Module):
    def __init__(self, num_classes=6, num_steps=25, beta=0.95, threshold=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.num_steps = num_steps
        spike_grad = surrogate.fast_sigmoid(slope=25)

        self.conv1 = nn.Conv1d(6, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm1d(32)
        self.lif1  = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm1d(64)
        self.lif2  = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn3   = nn.BatchNorm1d(128)
        self.lif3  = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool3 = nn.MaxPool1d(2)

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1     = nn.Linear(128, 64)
        self.lif4    = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.fc2     = nn.Linear(64, num_classes)
        self.lif_out = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()
        mem_out = self.lif_out.init_leaky()
        spk_rec = []

        for _ in range(self.num_steps):
            cur1 = self.bn1(self.conv1(x));   spk1, mem1   = self.lif1(cur1, mem1);   spk1 = self.pool1(spk1)
            cur2 = self.bn2(self.conv2(spk1)); spk2, mem2   = self.lif2(cur2, mem2);   spk2 = self.pool2(spk2)
            cur3 = self.bn3(self.conv3(spk2)); spk3, mem3   = self.lif3(cur3, mem3);   spk3 = self.pool3(spk3)
            spk3    = self.global_pool(spk3).squeeze(-1)
            cur4    = self.fc1(spk3);          spk4, mem4   = self.lif4(cur4, mem4)
            cur_out = self.fc2(spk4);          spk_out, mem_out = self.lif_out(cur_out, mem_out)
            spk_rec.append(spk_out)

        return torch.stack(spk_rec)  # [num_steps, batch, num_classes]


class SNNInferenceWrapper(nn.Module):
    """Static wrapper: input [batch, 6, 200] -> output [batch, 6] spike sums"""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x).sum(dim=0)  # sum over time steps


# ─── Load model ───────────────────────────────────────────────────────────────
print("Loading snn_fold_5.pth...")
device = torch.device("cpu")

base_model = MicroCNN_SNN(num_classes=6, num_steps=25, beta=0.95, threshold=1.0)
base_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
base_model.eval()

wrapper = SNNInferenceWrapper(base_model).to(device)
wrapper.eval()

dummy_input = torch.randn(1, 6, 200)
with torch.no_grad():
    out = wrapper(dummy_input)

print(f"Model loaded")
print(f"   Output shape: {out.shape}")
print(f"   Predicted class: {out.argmax(dim=1).item()}")

if ONNX_PATH.exists():
    ONNX_PATH.unlink()

print("Exporting to ONNX...")
torch.onnx.export(
    wrapper,
    dummy_input,
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["spike_sum"],
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
print(f"ONNX saved: {ONNX_PATH}")

# Verify with onnxruntime
sess = ort.InferenceSession(str(ONNX_PATH))
result = sess.run(None, {"input": dummy_input.numpy()})
print(f"ONNX verified")
print(f"   Output shape: {result[0].shape}")
print(f"   Predicted class: {result[0].argmax()}")
# %%
# ─── ONNX -> TFLite ───────────────────────────────────────────────────────────
import subprocess, sys

print("Converting ONNX -> TF SavedModel...")
result = subprocess.run(
    [sys.executable, "-m", "onnx2tf",
     "-i", str(ONNX_PATH),
     "-o", str(OUTPUT_DIR / "savedmodel"),
     "-osd"],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("onnx2tf failed:")
    print(result.stderr[-1000:])
else:
    print("SavedModel created")

    import tensorflow as tf

    # float32 TFLite
    converter = tf.lite.TFLiteConverter.from_saved_model(str(OUTPUT_DIR / "savedmodel"))
    tflite_model = converter.convert()
    TFLITE_PATH.write_bytes(tflite_model)
    print(f"TFLite float32: {TFLITE_PATH} ({len(tflite_model)/1024:.1f} KB)")

    # int8 quantized TFLite
    converter_int8 = tf.lite.TFLiteConverter.from_saved_model(str(OUTPUT_DIR / "savedmodel"))
    converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]

    def representative_dataset():
        for _ in range(100):
            yield [np.random.randn(1, 6, 200).astype(np.float32)]

    converter_int8.representative_dataset = representative_dataset
    converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter_int8.inference_input_type  = tf.int8
    converter_int8.inference_output_type = tf.int8

    tflite_int8 = converter_int8.convert()
    TFLITE_INT8.write_bytes(tflite_int8)
    print(f"TFLite int8: {TFLITE_INT8} ({len(tflite_int8)/1024:.1f} KB)")

# %%
# ─── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("DEPLOYMENT SUMMARY")
print("="*60)
print(f"ONNX:          {ONNX_PATH}")
print(f"TFLite fp32:   {TFLITE_PATH}")
print(f"TFLite int8:   {TFLITE_INT8}")
print()
print("Input:  [1, 6, 200] float32")
print("        channels: accel_x/y/z, gyro_x/y/z")
print("Output: [1, 6] spike counts -> argmax = class")
print()
classes = ["Fall_Initiation", "Pre_Fall", "Post_Fall", "ADL", "Near_Fall", "Recovery"]
print("Classes:")
for i, c in enumerate(classes):
    print(f"  {i}: {c}")
print()
print("Next: upload snn_fall_int8.tflite to Edge Impulse")
print("      -> Deployment -> Arduino library -> Nano BLE Sense")

In [ ]:
X_data = np.load(processed_dir / "X_data.npy")
y_labels = np.load(processed_dir / "y_labels.npy")

# Merge Impact and Aftermath
y_labels[y_labels == 7] = 6

# Remove Fall_Recovery (class 4)
mask = y_labels != 4
X_data = X_data[mask]
y_labels_temp = y_labels[mask]
y_labels = y_labels_temp.copy()
y_labels[y_labels_temp > 4] -= 1

print(f"X_data: {X_data.shape}")   # should be (16732, 200, 6)
print(f"y_labels: {y_labels.shape}")  # should be (16732,)

In [ ]:
import tensorflow as tf
import numpy as np

# Load TFLite int8 model
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_INT8))
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:", input_details[0]['dtype'], input_details[0]['shape'])
print("Output details:", output_details[0]['dtype'], output_details[0]['shape'])


X_data = np.load(processed_dir / "X_data.npy")
y_labels = np.load(processed_dir / "y_labels.npy")

# Run both models on same data
onnx_preds   = []
tflite_preds = []

# int8 quantization params
input_scale, input_zero_point = input_details[0]['quantization']

for i in range(len(X_data)):
    sample = X_data[i:i+1].astype(np.float32)  # [1, 200, 6] - original

    # ONNX inference - needs [1, 6, 200]
    onnx_out = sess.run(None, {"input": sample.transpose(0, 2, 1)})[0]
    onnx_preds.append(onnx_out.argmax())

    # TFLite int8 inference - needs [1, 200, 6]
    sample_int8 = (sample / input_scale + input_zero_point).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], sample_int8)
    interpreter.invoke()
    tflite_out = interpreter.get_tensor(output_details[0]['index'])
    tflite_preds.append(tflite_out.argmax())
    
onnx_preds   = np.array(onnx_preds)
tflite_preds = np.array(tflite_preds)

onnx_acc   = (onnx_preds == y_labels).mean()
tflite_acc = (tflite_preds == y_labels).mean()
agreement  = (onnx_preds == tflite_preds).mean()

print(f"\nONNX accuracy:        {onnx_acc:.4f} ({onnx_acc*100:.2f}%)")
print(f"TFLite int8 accuracy: {tflite_acc:.4f} ({tflite_acc*100:.2f}%)")
print(f"Model agreement:      {agreement:.4f} ({agreement*100:.2f}%)")
print(f"Quantization drop:    {(onnx_acc - tflite_acc)*100:.2f}%")

In [ ]:
print(input_details[0]['shape'])
print(input_details[0]['dtype'])

In [ ]:
# ONNX inference - channels first [1, 6, 200]
sample_cf = X_data[i:i+1].astype(np.float32).transpose(0, 2, 1)
onnx_out = sess.run(None, {"input": sample_cf})[0]
onnx_preds.append(onnx_out.argmax())

# TFLite int8 inference - channels last [1, 200, 6]
sample_cl = X_data[i:i+1].astype(np.float32)  # already [1, 200, 6]
sample_int8 = (sample_cl / input_scale + input_zero_point).astype(np.int8)
interpreter.set_tensor(input_details[0]['index'], sample_int8)
interpreter.invoke()
tflite_out = interpreter.get_tensor(output_details[0]['index'])
tflite_preds.append(tflite_out.argmax())

In [ ]:
import os
print(os.getcwd())
print(torch.__version__)

In [ ]:
# ─── Export to ONNX ───────────────────────────────────────────────────────────
import os
import onnxruntime as ort

if ONNX_PATH.exists():
    ONNX_PATH.unlink()

print("Exporting to ONNX...")
exported = torch.onnx.dynamo_export(wrapper, dummy_input)
exported.save(str(ONNX_PATH))
print(f"✅ ONNX saved: {ONNX_PATH}")

# Verify with onnxruntime
sess = ort.InferenceSession(str(ONNX_PATH))
result = sess.run(None, {"input": dummy_input.numpy()})
print(f"✅ ONNX verified")
print(f"   Output shape: {result[0].shape}")
print(f"   Predicted class: {result[0].argmax()}")

In [ ]:
sess = ort.InferenceSession(str(ONNX_PATH))
dummy = np.random.randn(1, 6, 200).astype(np.float32)
result = sess.run(None, {"input": dummy})
print(f"✅ ONNX runtime works")
print(f"Spike sums: {result[0]}")
print(f"Predicted class: {result[0].argmax()}")